# AMR surveillance data standardization — Tunisian hospitals (GLASS / SIRSCAN / SantéLab / WHONET / VITEK / IMKO)

**Goal** — turn 100 heterogeneous hospital exports into **one auditable, English-language, analysis-ready dataset**, with every step
verified by an explicit, quantified test (row conservation, coverage with confidence intervals, agreement and kappa between mapping routes,
hold-out validation, monotonicity of S/I/R calls).

**Design principles (read once)**

1. **`df_long` is the master table** (one row = one isolate × one antibiotic result). Wide tables are *derived views*, never the source of truth.
2. **Nothing disappears silently.** Every dropped row/cell goes to an `excluded` ledger with a reason, and conservation equations are asserted.
3. **Raw strings are data, standard values are English.** Dictionaries keep the French *patterns* (they match French source text) and carry English *targets*.
4. **Two independent mapping routes are always compared** (your dictionary vs. the `AMR` package); disagreements are triaged, never overwritten silently.
5. **`method` is decided from the cell value, not from the column header** (see section 6: headers lie).
6. **`as_sir` has no dose argument** (verified in section 9) — dose is used only to *infer which guideline the lab applied*.
7. **Privacy by design**: patient names are never loaded into the output; patient IDs are salted-hashed; only birth *year* is kept.

Run order: top to bottom. Environment variables: `AMR_BASE`, `AMR_DATA`, `AMR_DICT`, `AMR_OUT`, `AMR_SALT`.

In [ ]:
import os, re, io, sys, json, math, hashlib, unicodedata, warnings, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_rows", 120)

BASE      = Path(os.environ.get("AMR_BASE", Path.cwd()))
DATA_DIR  = Path(os.environ.get("AMR_DATA", BASE / "copie_files_no_patient"))
DICT_DIR  = Path(os.environ.get("AMR_DICT", BASE / "dictionaries_en"))
OUT_DIR   = Path(os.environ.get("AMR_OUT",  BASE / "outputs"))
AUDIT_DIR = OUT_DIR / "audit"
for d in (OUT_DIR, AUDIT_DIR): d.mkdir(parents=True, exist_ok=True)

SEED = 20261007
rng  = np.random.default_rng(SEED)
KEEP_BIRTH_DATE = False                       # data minimisation: keep birth YEAR only
PSEUDO_SALT = os.environ.get("AMR_SALT")
if not PSEUDO_SALT:
    PSEUDO_SALT = "CHANGE-ME-" + str(SEED)
    print("WARNING: AMR_SALT not set -> using a default salt. Set a secret salt before sharing any output.")

print("DATA_DIR :", DATA_DIR, "| exists:", DATA_DIR.exists())
print("DICT_DIR :", DICT_DIR, "| exists:", DICT_DIR.exists())
print("CSV files:", len(list(DATA_DIR.glob('*.csv'))))

In [ ]:
# ---- AMR package (R backend through rpy2). The offline test double is ONLY for CI without R. ----
AMR, AMR_OK, AMR_IS_FAKE = None, False, False
if os.environ.get("AMR_FAKE") == "1":
    AMR_IS_FAKE = True          # defined in the appendix cell at the bottom (exec'd below)
else:
    try:
        import AMR as _AMR
        AMR, AMR_OK = _AMR, True
        AMR.mo_reset_session()
        print("AMR python wrapper loaded.")
    except Exception as e:
        print("AMR package NOT available -> AMR-dependent sections will be skipped:", repr(e)[:200])

In [ ]:
# ---- OFFLINE TEST DOUBLE (only when AMR_FAKE=1; used to test the pipeline code without R). NOT the AMR package: its results are meaningless. ----
if AMR_IS_FAKE:
    class _FakeAMR:
        _AB = {"amikacin":"AMK","amoxicillin":"AMX","amoxicillin/clavulanic acid":"AMC","ampicillin":"AMP","ampicillin/sulbactam":"SAM","aztreonam":"ATM","cefalexin":"LEX",
               "cefalotin":"CEP","cefepime":"FEP","cefixime":"CFM","cefotaxime":"CTX","cefoxitin":"FOX","ceftazidime":"CAZ","ceftriaxone":"CRO","cefuroxime":"CXM",
               "chloramphenicol":"CHL","ciprofloxacin":"CIP","clindamycin":"CLI","colistin":"COL","doxycycline":"DOX","ertapenem":"ETP","erythromycin":"ERY",
               "fosfomycin":"FOS","fusidic acid":"FUS","gentamicin":"GEN","imipenem":"IPM","kanamycin":"KAN","levofloxacin":"LVX","linezolid":"LNZ","meropenem":"MEM",
               "mecillinam":"MEC","minocycline":"MNO","moxifloxacin":"MFX","nalidixic acid":"NAL","netilmicin":"NET","nitrofurantoin":"NIT","norfloxacin":"NOR",
               "ofloxacin":"OFX","oxacillin":"OXA","benzylpenicillin":"PEN","piperacillin":"PIP","piperacillin/tazobactam":"TZP","rifampicin":"RIF","teicoplanin":"TEC",
               "tetracycline":"TCY","ticarcillin":"TIC","ticarcillin/clavulanic acid":"TCC","tigecycline":"TGC","tobramycin":"TOB","trimethoprim/sulfamethoxazole":"SXT",
               "vancomycin":"VAN","streptomycin":"STR","temocillin":"TEM","mupirocin":"MUP","quinupristin/dalfopristin":"QDA","pristinamycin":"PRI","cefpodoxime":"CPD",
               "trimethoprim":"TMP","azithromycin":"AZM","clarithromycin":"CLR","cefazolin":"CZO"}
        _MO = ["Escherichia coli","Klebsiella pneumoniae","Enterobacter cloacae","Salmonella","Shigella","Streptococcus pneumoniae","Enterococcus faecium",
               "Enterococcus faecalis","Acinetobacter baumannii","Pseudomonas aeruginosa","Staphylococcus aureus","Neisseria gonorrhoeae","Haemophilus influenzae",
               "Streptococcus pyogenes","Streptococcus agalactiae","Neisseria meningitidis","Proteus mirabilis","Mycobacterium tuberculosis"]
        _MO_ALIAS = {"e coli":"Escherichia coli","e.coli":"Escherichia coli","staph aureus":"Staphylococcus aureus","m catar":"Mycobacterium tuberculosis"}  # last one = a deliberate wrong match
        @staticmethod
        def _k(x): return re.sub(r"[^a-z0-9/.]+", " ", str(x).lower()).strip()
        def as_ab(self, x):
            inv = {v: v for v in self._AB.values()}
            return [self._AB.get(self._k(v)) or inv.get(str(v).strip().upper()) for v in x]
        def ab_name(self, x):
            rev = {v: k.capitalize() for k, v in self._AB.items()}
            return [rev.get(c) or (self._k(c).capitalize() if self._k(c) in self._AB else None) for c in x]
        def _mo(self, v):
            k = self._k(v); low = {m.lower(): m for m in self._MO}
            return self._MO_ALIAS.get(k) or low.get(k)
        def as_mo(self, x): return [("B_" + m.split()[0][:5].upper() + "_" + (m.split() + ["SPP"])[1][:4].upper()) if (m := self._mo(v)) else None for v in x]
        def mo_name(self, x): return [self._mo(v) for v in x]
        def mo_uncertainties(self): return []
        def mo_reset_session(self): pass
        def add_custom_microorganisms(self, df): return None
        def add_custom_antimicrobials(self, df): return None
        @property
        def clinical_breakpoints(self):
            p = os.environ.get("AMR_BP_PICKLE", "/tmp/amrsrc/clinical_breakpoints.pkl")
            return pd.read_pickle(p) if os.path.exists(p) else pd.DataFrame(columns=["guideline", "type", "method", "ab", "disk_dose", "breakpoint_S", "breakpoint_R"])
        def as_sir(self, x, mo=None, ab=None, guideline="EUCAST 2020", uti=None, enforce_method="auto", **kw):
            bp = self.clinical_breakpoints; m = "DISK" if enforce_method == "disk" else "MIC"
            r = bp[(bp.guideline == guideline) & (bp.ab == ab) & (bp.method == m) & bp.breakpoint_S.notna()]
            if r.empty: return [None] * len(x)
            S, R = float(r.iloc[0].breakpoint_S), float(r.iloc[0].breakpoint_R); clsi = guideline.startswith("CLSI"); out = []
            for v in x:
                n = float(re.sub(r"[^0-9.]", "", str(v)) or "nan")
                if m == "DISK": out.append("S" if n >= S else ("R" if (n <= R if clsi else n < R) else "I"))
                else: out.append("S" if n <= S else ("R" if (n >= R if clsi else n > R) else "I"))
            return out
        def sir_interpretation_history(self, clean=False): return pd.DataFrame(columns=["outcome"])
    AMR = _FakeAMR(); AMR_OK = False
    print("USING OFFLINE TEST DOUBLE - results are NOT scientific.")
AMR_READY = AMR_OK or AMR_IS_FAKE

In [ ]:
# ---- test registry + statistics helpers (used by every section) ----
TESTS = []
def check(name, ok, detail=""):
    TESTS.append({"test": name, "pass": bool(ok), "detail": str(detail)})
    print(("PASS " if ok else "FAIL ") + name + (f"  [{detail}]" if detail != "" else ""))
    return bool(ok)

def wilson(k, n, alpha=0.05):
    if n == 0: return (np.nan, np.nan)
    z = stats.norm.ppf(1 - alpha / 2); p = k / n
    den = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return (max(0, c - h), min(1, c + h))

def pct_ci(k, n):
    lo, hi = wilson(k, n)
    return f"{100*k/max(n,1):.2f}% (95% CI {100*lo:.2f}-{100*hi:.2f}; n={n})"

def cohen_kappa(a, b):
    a, b = pd.Series(a).astype(str).values, pd.Series(b).astype(str).values
    labs = sorted(set(a) | set(b)); idx = {l: i for i, l in enumerate(labs)}
    m = np.zeros((len(labs), len(labs)))
    for x, y in zip(a, b): m[idx[x], idx[y]] += 1
    n = m.sum()
    if n == 0: return np.nan
    po = np.trace(m) / n; pe = (m.sum(0) * m.sum(1)).sum() / n**2
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

def mcnemar_exact(b, c):
    # discordant pairs b (A ok, B not) vs c (B ok, A not); exact two-sided binomial
    n = b + c
    return 1.0 if n == 0 else stats.binomtest(min(b, c), n, 0.5).pvalue

def fisher_two_prop(k1, n1, k2, n2):
    return stats.fisher_exact([[k1, n1 - k1], [k2, n2 - k2]])[1]

def exact_binom_geq(k, n, p0):
    # H0: true proportion <= p0 ; H1: proportion > p0
    return stats.binomtest(k, n, p0, alternative="greater").pvalue
print("helpers ready")

In [ ]:
# ---- text utilities ----
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))

def fix_mojibake(s):
    # 'BÃŠTA' -> 'BÊTA', 'RÃ‰SISTANT' -> 'RÉSISTANT' (UTF-8 bytes read as cp1252/latin-1)
    if not isinstance(s, str) or not re.search("[ÃÂ]", s): return s
    for enc in ("cp1252", "latin-1"):
        try:
            t = s.encode(enc).decode("utf-8")
            if t != s: return t
        except Exception:
            pass
    return s

def norm_key(s):
    # accent/case/space-insensitive comparison key
    s = strip_accents(fix_mojibake(str(s))).lower().strip()
    return re.sub(r"\s+", " ", s)

def clean_header(h):
    return re.sub(r"\s+", " ", fix_mojibake(str(h))).strip()

def decode_file_label(name):
    # zip export wrote non-ASCII characters as '#U00e9'
    return re.sub(r"#U([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), name)

def read_csv_robust(path):
    raw = Path(path).read_bytes()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            txt = raw.decode(enc); break
        except UnicodeDecodeError:
            continue
    head = txt.split("\n", 1)[0]
    sep = max([",", ";", "\t", "|"], key=head.count)
    df = pd.read_csv(io.StringIO(txt), sep=sep, low_memory=False, dtype=object)
    df.columns = [clean_header(c) for c in df.columns]
    return df, enc, sep
print("utilities ready")

## 1. File inventory, hospital / year inference, duplicate detection

*Test*: every file must resolve to **exactly one hospital** and **one family**; byte-identical files are removed **before** any statistics
(duplicated files silently inflate resistance counts).

In [ ]:
HOSP = pd.read_excel(DICT_DIR / "hospitals_en.xlsx")
HOSP_FR = dict(zip(HOSP.hospital_code, HOSP.hospital_fr)); HOSP_EN = dict(zip(HOSP.hospital_code, HOSP.hospital_en))

# ordered rules: first match wins, evaluated on the file name BEFORE the sheet name ('__'), case-insensitive, token-bounded.
# confidence: H = hospital named in the file name; L = inferred from a naming convention (CONFIRM with the lab)
B0, B1 = r"(?<![A-Za-z])", r"(?![A-Za-z])"
HOSP_RULES = [
    (r"CTGB", "CTGB", "H"), (r"CNGMO", "CNGMO", "H"), (r"IMKO", "IMKO", "H"),
    (r"A\.?MAMI|ARIANA", "AMAMI", "H"),
    (B0 + r"HAO" + B1 + r"|AZIZA", "HAO", "H"),
    (B0 + r"HCN" + B1 + r"|CHARLES", "HCN", "H"),
    (r"MONASTIR|FATTOULMA", "CHUM", "H"),
    (r"RABTA", "RABTA", "H"),
    (r"SOUSSE|HACHED|HACED", "FHS", "H"),
    (B0 + r"(HEBHT|HET|HE)" + B1 + r"|D.ENFANTS|HOP[_ ]ENF", "HE", "H"),
    (r"STAT[-_]WHONET|STAT[-_]SANT", "FHS", "L"),        # naming convention shared with the Sousse 2022/2023/2024 files
    (r"ATB_GLASS_2021|^ATB_|VITEK", "HAO", "L"),         # VITEK exports are named this way only for HAO
]
MANUAL_HOSPITAL = {}   # e.g. {"W20TUN__W20TUN.csv": "HCN"}  <- fill after confirmation with the data owner
def infer_hospital(fname, label):
    if fname in MANUAL_HOSPITAL: return MANUAL_HOSPITAL[fname], "M"
    stem = label.split("__")[0]
    for pat, code, conf in HOSP_RULES:
        if re.search(pat, stem, flags=re.I): return code, conf
    return "UNKNOWN", "-"

def infer_year(name):
    ys = re.findall(r"(?<!\d)(20(?:19|2[0-6]))(?!\d)", name)
    return int(ys[0]) if ys else None

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

rows = []
for p in sorted(DATA_DIR.glob("*.csv")):
    label = decode_file_label(p.stem)
    code, conf = infer_hospital(p.name, label)
    rows.append(dict(file=p.name, label=label, hospital_code=code, hosp_conf=conf, year=infer_year(label),
                     size_kb=round(p.stat().st_size / 1024), sha256=sha256_file(p)))
INV = pd.DataFrame(rows)
INV["file_id"] = [f"F{i:03d}" for i in range(len(INV))]
print(INV.groupby("hospital_code").size().to_string())
unk = INV[INV.hospital_code == "UNKNOWN"]
print("\nFiles with UNKNOWN hospital:", len(unk)); print(unk[["file", "year"]].to_string())
print("\nLow-confidence hospital inference (confirm with the lab):"); print(INV[INV.hosp_conf == "L"][["file", "hospital_code"]].to_string())
print("\nFiles without year:", INV[INV.year.isna()].file.tolist())
check("every file resolves to a known hospital", (INV.hospital_code != "UNKNOWN").all(), f"{(INV.hospital_code=='UNKNOWN').sum()} unresolved")

In [ ]:
dups = INV[INV.duplicated("sha256", keep=False)].sort_values("sha256")
print("Byte-identical file groups:"); print(dups[["file_id", "file", "sha256"]].assign(sha256=lambda d: d.sha256.str[:10]).to_string())
check("no byte-identical duplicate files", dups.empty, f"{dups.sha256.nunique()} groups / {len(dups)} files")
# keep the first file of each group; the others are logged and excluded
INV["is_duplicate_file"] = INV.duplicated("sha256", keep="first")
MAXF = int(os.environ.get("AMR_MAX_FILES", "0"))      # CI/smoke-test switch only: process every k-th file (leave unset for the real run)
if MAXF:
    step = max(1, len(INV) // MAXF); INV["is_duplicate_file"] |= (INV.index % step != 0); print(f"SMOKE-TEST MODE: {int((~INV.is_duplicate_file).sum())} files kept")
INV.to_csv(AUDIT_DIR / "01_inventory.csv", index=False)
print("files kept:", (~INV.is_duplicate_file).sum(), "| duplicate files excluded:", INV.is_duplicate_file.sum())

## 2. Load the English dictionaries and audit them *before* trusting them

Dictionary defects are the #1 silent source of error in AMR pipelines. Three automatic audits:
**(a)** every regex compiles, **(b)** no pattern maps to two different targets (ambiguity), **(c)** structural findings
(non-antibiotic entries that must never reach `as_sir`: MDR/ESBL flags, Vitek `Family -` aggregate columns, `Unknown`).

In [ ]:
from functools import lru_cache
DN = ["atb_mapping", "cmi_mapping", "standard_mapping", "service_by_hospital", "specimen_by_hospital",
      "mo_synonym_map", "breakpoints_by_organism", "ab_translation"]
D = {n: pd.read_excel(DICT_DIR / f"{n}_en.xlsx", dtype=object) for n in DN}
for n, d in D.items(): print(f"{n:26s} {d.shape}")

class RegexDict:
    def __init__(self, df, name):
        self.df, self.name, self.bad, recs = df.reset_index(drop=True), name, [], []
        for i, pat in enumerate(self.df["pattern"].astype(str)):
            try: rx = re.compile(pat, re.I)
            except re.error as e: self.bad.append((i, pat, str(e))); continue
            wildcard = bool(re.search(r"\.\*|\.\+|\.\{|\\s\*\.", pat))
            unanchored = not (pat.startswith("^") and pat.endswith("$"))
            recs.append((wildcard, unanchored, -len(pat), i, rx))
        # most specific first: anchored-exact, then unanchored, wildcard last; longer pattern first; dictionary order breaks ties
        self.rx = [(i, rx) for _, _, _, i, rx in sorted(recs, key=lambda t: t[:4])]
    def matches(self, text): return [i for i, rx in self.rx if rx.search(text)]


# ---- documented corrections to YOUR dictionaries (review and approve each line; they are applied in memory, originals untouched) ----
FIX_LOG = []
def _drop(df, mask, why):
    for _, r in df[mask].iterrows(): FIX_LOG.append(dict(action="drop", pattern=r["pattern"], target=str(r.get("standard_column_en", r.get("antibiotic_en"))), why=why))
    return df[~mask].reset_index(drop=True)
SM = D["standard_mapping"]
SM = _drop(SM, (SM.pattern == "^Lab ID$") & (SM.standard_column_en == "patient_id"), "'Lab ID' is the sample identifier in VITEK exports; 'Patient ID' is a separate column")
SM = _drop(SM, (SM.pattern == "^Organism Code$") & (SM.standard_column_en == "organism_raw"), "a code is not an organism name (duplicate pattern -> organism_code)")
m_ = SM.pattern == "^Num[ée]ro de pr[ée]l[èe]vement$"
for _, r in SM[m_].iterrows(): FIX_LOG.append(dict(action="retarget", pattern=r.pattern, target="patient_id -> specimen_number", why="values like '1306-1307' are specimen numbers, not patient ids"))
SM.loc[m_, "standard_column_en"] = "specimen_number"
for pat, tgt, why in [(r"^Patient ID$", "patient_id", "header absent from dictionary"), (r"^Hopital$", "hospital_raw", "header absent from dictionary"),
                      (r"^Date de naiss\.?$", "birth_date", "safety net")]:
    if not (SM.pattern == pat).any():
        SM = pd.concat([SM, pd.DataFrame([dict(pattern=pat, standard_column_fr="(added)", standard_column_en=tgt)])], ignore_index=True)
        FIX_LOG.append(dict(action="add", pattern=pat, target=tgt, why=why))
D["standard_mapping"] = SM
AT = D["atb_mapping"]
AT = _drop(AT, (AT.pattern == "^PRL100$") & (AT.antibiotic_en == "Pristinamycin"), "PRL = piperacillin (WHONET); pristinamycin is PRI; the same pattern pointed to both")
for pat, dose, why in [(r"^CXM$", "", "WHONET code CXM = cefuroxime (cells hold S/I/R and disk mm), was filed as 'Unknown'"),
                       (r"^CXM_FD30$", "30", "WHONET-style cefuroxime 30 µg disk, was filed as 'Unknown'"), (r"^cxm_fd30$", "30", "idem")]:
    AT = _drop(AT, (AT.pattern == pat) & (AT.result_kind == "unknown_column"), why)
    AT = pd.concat([AT, pd.DataFrame([dict(pattern=pat, standard_column_fr="cefuroxime", result_kind="antibiotic", antibiotic_en="Cefuroxime",
                                           antibiotic_class_en=None, dose_ug=dose, method_hint="disk" if dose else "unknown")])], ignore_index=True)
    FIX_LOG.append(dict(action="add", pattern=pat, target="Cefuroxime", why=why))
D["atb_mapping"] = AT
PII_RX = re.compile(r"^(nom( du patient)?|patient|pr[ée]nom|first_?name|last_?name|nom_?patient)$", re.I)
print(pd.DataFrame(FIX_LOG).to_string(index=False))

META, ATB, CMI = (RegexDict(D["standard_mapping"], "standard_mapping"), RegexDict(D["atb_mapping"], "atb_mapping"),
                  RegexDict(D["cmi_mapping"], "cmi_mapping"))
for r in (META, ATB, CMI): check(f"all regex compile in {r.name}", not r.bad, r.bad[:2])

# structure findings
print("\natb_mapping result kinds :", D["atb_mapping"].result_kind.value_counts().to_dict())
print("cmi_mapping result kinds :", D["cmi_mapping"].result_kind.value_counts().to_dict())
dup_pat = D["atb_mapping"][D["atb_mapping"].duplicated("pattern", keep=False)].sort_values("pattern")
conf = dup_pat.groupby("pattern").antibiotic_en.nunique()
CONFLICT_PATTERNS = list(conf[conf > 1].index)      # tested against the OBSERVED headers in section 3
print("identical patterns that point to different antibiotics:", CONFLICT_PATTERNS)
print(dup_pat[dup_pat.pattern.isin(conf[conf > 1].index)][["pattern", "standard_column_fr", "antibiotic_en"]].to_string())
print("\nstandard-value inconsistencies fixed in the English dictionaries:")
print(" service  : 'Dispensaire'/'Dispensaires', 'Réanimation' vs 'Réanimation (adulte et enfant)' -> one ward + separate `population` field")
print(" specimen : 'Coproculture' vs 'Selles' -> single 'Stool'; sentinel 'Non' -> 'EXCLUDED_non_target_*' (explicit, no ambiguity with 'No')")

In [ ]:
# ---- antibiotic header parsing (name / dose / method hint) ----
def norm_dose(d):
    d = str(d).replace(",", ".").replace("/", "-")
    d = re.sub(r"\s+", "", d)
    d = re.sub(r"(\d+)\.0+(?=-|$)", r"\1", d)
    return d

def parse_ab_header(h):
    h0 = re.sub(r"\.\d+$", "", clean_header(h))                 # pandas duplicate suffix 'AMP.1'
    is_mic = bool(re.search(r"\(\s*CMI\s*\)|\bCMI\b|\bMIC\b", h0, re.I))
    is_int = bool(re.search(r"interpr", h0, re.I))
    name = re.sub(r"\(\s*CMI\s*\)|\bCMI\b|\bMIC\b|interpr[ée]tation", "", h0, flags=re.I)
    m = re.search(r"(\d+(?:[.,]\d+)?(?:\s*[-/]\s*\d+(?:[.,]\d+)?)?)\s*(µg|ug|mcg|UI|U)?\s*$", name)
    dose = ""
    if m and m.start() > 0:
        dose = norm_dose(m.group(1)); name = name[:m.start()]
    name = re.sub(r"[\s\-_/]+$", "", name).strip()
    hint = "sir" if is_int else ("mic" if is_mic else ("disk" if dose else "unknown"))
    return name, dose, hint

# regression tests on REAL header shapes seen in the files
CASES = {"AMPICILLINE 2µg": ("AMPICILLINE", "2", "disk"), "AMPICILLINE (CMI)": ("AMPICILLINE", "", "mic"),
         "AMOXICILLINE + AC.CLAVULANIQUE 20-10µg": ("AMOXICILLINE + AC.CLAVULANIQUE", "20-10", "disk"),
         "TRIMETHOPRIME + SULFAMIDES 1,25-23,75µg": ("TRIMETHOPRIME + SULFAMIDES", "1.25-23.75", "disk"),
         "PIPERACILLINE 100": ("PIPERACILLINE", "100", "disk"), "CEFTAZIDIME": ("CEFTAZIDIME", "", "unknown"),
         "GEH.1": ("GEH", "", "unknown"), "Acide fusidique Interprétation": ("Acide fusidique", "", "sir")}
bad = {k: (parse_ab_header(k), v) for k, v in CASES.items() if parse_ab_header(k) != v}
check("parse_ab_header regression cases", not bad, bad)

AB_NAME_INDEX = {}
for _, r in D["ab_translation"].iterrows():
    AB_NAME_INDEX[norm_key(r.base_key_fr.replace("_", " "))] = r.antibiotic_en
    AB_NAME_INDEX[norm_key(r.antibiotic_en)] = r.antibiotic_en
for k, v in {"furanes": "Nitrofurantoin", "peni g": "Benzylpenicillin", "penicilline g": "Benzylpenicillin",
             "trimethoprime sulfamides": "Trimethoprim/sulfamethoxazole", "cotrimoxazole": "Trimethoprim/sulfamethoxazole"}.items():
    AB_NAME_INDEX[k] = v

@lru_cache(maxsize=None)
def resolve_header(h):
    h = re.sub(r"\.\d+$", "", clean_header(h))
    if PII_RX.match(h):
        return dict(kind="pii_drop", target="patient_name", antibiotic_en=None, dose="", method_hint="", rule="pii", n_cand=1, alts="")
    mm = META.matches(h)
    if mm:
        tg = sorted({META.df.standard_column_en[i] for i in mm})
        return dict(kind="metadata", target=META.df.standard_column_en[mm[0]], antibiotic_en=None, dose="", method_hint="",
                    rule="standard_mapping", n_cand=len(tg), alts="|".join(tg))
    hint_cmi = bool(re.search(r"cmi|\bmic\b|interpr", h, re.I))
    for dd in ([CMI, ATB] if hint_cmi else [ATB, CMI]):
        mm = dd.matches(h)
        if mm:
            d = dd.df; i0 = mm[0]
            tg = sorted({f"{d.result_kind[i]}:{d.antibiotic_en[i] if pd.notna(d.antibiotic_en[i]) else d.antibiotic_class_en[i]}" for i in mm})
            _, pdose, phint = parse_ab_header(h)
            dose = (d.dose_ug[i0] if pd.notna(d.dose_ug[i0]) and str(d.dose_ug[i0]) != "" else pdose)
            mh = d.method_hint[i0] if d.method_hint[i0] not in (None, "unknown") else phint
            return dict(kind=d.result_kind[i0], target=d.standard_column_fr[i0],
                        antibiotic_en=d.antibiotic_en[i0] if pd.notna(d.antibiotic_en[i0]) else None,
                        dose=str(dose) if pd.notna(dose) else "", method_hint=mh, rule=dd.name, n_cand=len(tg), alts="|".join(tg))
    name, dose, hint = parse_ab_header(h)
    cands = [name] + ([h.split("-", 1)[1]] if re.match(r"^[A-Za-z0-9]{1,5}-.+", h) else [])
    for cnd in cands:
        en = AB_NAME_INDEX.get(norm_key(cnd).replace("_", " "))
        if en: return dict(kind="antibiotic", target=name, antibiotic_en=en, dose=dose, method_hint=hint,
                           rule="fallback_name", n_cand=1, alts=en)
    return dict(kind="unmapped", target=None, antibiotic_en=None, dose=dose, method_hint=hint, rule="none", n_cand=0, alts="")

for h in ["Numéro de patient", "Germe", "AMPICILLINE 2µg", "Acide fusidique CMI", "BMR", "Family -AMINOSIDES - 1", "ZZZ-unknown drug"]:
    print(f"{h!r:34} ->", {k: v for k, v in resolve_header(h).items() if k in ('kind','target','antibiotic_en','dose','method_hint','rule')})

## 3. File families (the files are NOT homogeneous) and per-file column mapping

Five structurally different export types exist; each is processed **on its own** (per-file mapping table saved to `audit/colmap_<file>.csv`).
`SANTELAB` files are laboratory-information-system extracts (one row per *test*, **no antibiotic results**) → they feed `lis_records`, not `df_long`.

In [ ]:
def classify_family(fname, cols):
    cl = [c.lower() for c in cols]; n = len(cols)
    if any("paramètre" in c or "parametre" in c for c in cl) and n <= 14: return "SANTELAB_LIS"
    if any(c.startswith("micro-organisme") for c in cl): return "WHONET_SIR"
    if sum(bool(re.match(r"^[A-Za-z0-9]{1,5}-[A-Za-zÀ-ÿ]", c)) for c in cols) >= 15 or any(c.startswith("family -") for c in cl): return "VITEK"
    if re.search(r"IMKO", fname, re.I): return "IMKO_LABPRO"
    return "SIRSCAN_WIDE"

REG = {}   # file_id -> dict(df=None, colmap=..., family=...)
colmaps, fam_rows = [], []
t0 = time.time()
for r in INV[~INV.is_duplicate_file].itertuples():
    df, enc, sep = read_csv_robust(DATA_DIR / r.file)
    fam = classify_family(r.file, list(df.columns))
    rows = []
    for c in df.columns:
        rr = resolve_header(c); rows.append({**rr, "file_id": r.file_id, "col_raw": c, "n_nonnull": int(df[c].notna().sum())})
    cm = pd.DataFrame(rows); cm["family"] = fam
    cm.to_csv(AUDIT_DIR / f"colmap_{r.file_id}.csv", index=False)
    colmaps.append(cm)
    fam_rows.append(dict(file_id=r.file_id, family=fam, enc=enc, sep=sep, n_rows=len(df), n_cols=df.shape[1]))
    del df
COLMAP = pd.concat(colmaps, ignore_index=True)
FAM = pd.DataFrame(fam_rows); INV = INV.merge(FAM, on="file_id", how="left")
print(f"mapped headers of {INV.file_id.nunique()} files in {time.time()-t0:.0f}s")
print(INV.groupby("family").agg(files=("file_id", "count"), rows=("n_rows", "sum")).to_string())
print("\nencodings:", INV.enc.value_counts().to_dict(), "| separators:", INV.sep.value_counts().to_dict())

In [ ]:
# ---- coverage of the column mapping (cell-weighted) with Wilson CI, and ambiguity audit ----
cm = COLMAP[COLMAP.n_nonnull > 0].copy()
cm["resolved"] = cm.kind != "unmapped"
by_kind = cm.groupby("kind").agg(columns=("col_raw", "count"), cells=("n_nonnull", "sum")).sort_values("cells", ascending=False)
print(by_kind.to_string())
k, n = int(cm.loc[cm.resolved, "n_nonnull"].sum()), int(cm.n_nonnull.sum())
print("\ncell-weighted header coverage :", pct_ci(k, n))
k2, n2 = int(cm.resolved.sum()), len(cm)
print("column-level header coverage  :", pct_ci(k2, n2))
cov_file = cm.groupby("file_id").apply(lambda g: g.loc[g.resolved, "n_nonnull"].sum() / g.n_nonnull.sum())
print("per-file coverage: min %.3f | p05 %.3f | median %.3f" % (cov_file.min(), cov_file.quantile(.05), cov_file.median()))
check("header coverage >= 98% of non-empty cells", k / n >= 0.98, pct_ci(k, n))
hit = [p for p in CONFLICT_PATTERNS if any(re.search(p, h, re.I) for h in cm.col_raw.unique())]
check("contradictory dictionary patterns are not triggered by any observed header", not hit, hit)

un = (cm[~cm.resolved].groupby("col_raw").agg(files=("file_id", "nunique"), cells=("n_nonnull", "sum")).sort_values("cells", ascending=False))
un.to_csv(AUDIT_DIR / "03_unmapped_headers.csv"); print("\nTop unmapped headers (fix these first):"); print(un.head(25).to_string())

amb = cm[(cm.n_cand > 1)].drop_duplicates("col_raw")[["col_raw", "kind", "alts", "rule"]]
amb.to_csv(AUDIT_DIR / "03_ambiguous_headers.csv", index=False)
check("ambiguous headers (>1 distinct target) < 2% of headers", len(amb) / cm.col_raw.nunique() < 0.02,
      f"{len(amb)} / {cm.col_raw.nunique()} unique headers"); print(amb.head(15).to_string())
unk = cm[cm.kind.isin(["unknown_column", "panel_family_column", "phenotype_flag"])].groupby(["kind", "col_raw"]).n_nonnull.sum()
print("\nNon-antibiotic columns detected (excluded from as_sir by design):"); print(unk.sort_values(ascending=False).head(15).to_string())

In [ ]:
# ---- HOLD-OUT validation: does the dictionary behave the same on files it was not tuned on? ----
# NOTE: the dictionary was hand-built from these very files, so this tests *homogeneity across files*, not true generalisation.
# True out-of-sample test = apply to a NEW year (2025) and read the same two numbers below.
files = INV[~INV.is_duplicate_file].file_id.values
test_files = set(rng.choice(files, size=max(5, len(files) // 5), replace=False))
cov = cm.groupby("file_id").apply(lambda g: g.loc[g.resolved, "n_nonnull"].sum() / g.n_nonnull.sum()).rename("coverage")
a, b = cov[~cov.index.isin(test_files)], cov[cov.index.isin(test_files)]
u = stats.mannwhitneyu(a, b, alternative="two-sided")
print(f"train files n={len(a)} mean coverage={a.mean():.4f} | hold-out files n={len(b)} mean coverage={b.mean():.4f}")
print(f"Mann-Whitney U p-value = {u.pvalue:.4f}  (H0: same coverage distribution)")
boot = [rng.choice(b.values, len(b)).mean() - rng.choice(a.values, len(a)).mean() for _ in range(5000)]
lo_, hi_ = np.percentile(boot, [2.5, 97.5]); print(f"bootstrap 95% CI of (holdout - train) mean coverage: [{lo_:.4f}, {hi_:.4f}]")
check("hold-out coverage not significantly worse than train (p>0.05)", u.pvalue > 0.05, f"p={u.pvalue:.4f}")

## 4. Hospital cross-validation from file *content* (does the file name tell the truth?)

For each file we compare its observed ward / specimen strings with the per-hospital dictionaries: the hospital whose dictionary explains the most
distinct strings is the **content-based** hospital. Agreement with the file-name hospital is a direct, quantified test of the inference rules —
and it resolves files such as `W20TUN` that carry no hospital name.

In [ ]:
SV, SP = D["service_by_hospital"].copy(), D["specimen_by_hospital"].copy()
H_FR2CODE = {v: k for k, v in HOSP_FR.items()}
SV["h"] = SV.hospital.map(H_FR2CODE); SP["h"] = SP.hospital.map(H_FR2CODE)
SV["k"] = SV.raw_value.map(norm_key); SP["k"] = SP.raw_value.map(norm_key)
SV_KEYS = SV.groupby("h").k.apply(set).to_dict(); SP_KEYS = SP.groupby("h").k.apply(set).to_dict()
meta_cols = lambda cmf, tgt: cmf.loc[(cmf.kind == "metadata") & (cmf.target == tgt) & (cmf.n_nonnull > 0), "col_raw"].tolist()

def content_hospital(df, cmf):
    obs_s, obs_p = set(), set()
    for c in meta_cols(cmf, "ward_raw"): obs_s |= set(df[c].dropna().map(norm_key))
    for c in meta_cols(cmf, "specimen_raw"): obs_p |= set(df[c].dropna().map(norm_key))
    sc = {h: (len(obs_s & SV_KEYS.get(h, set())) / max(len(obs_s), 1) + len(obs_p & SP_KEYS.get(h, set())) / max(len(obs_p), 1)) / 2 for h in HOSP_FR}
    best = sorted(sc.items(), key=lambda t: -t[1])
    return best[0][0], best[0][1], best[1][1]

rows = []
for r in INV[~INV.is_duplicate_file].itertuples():
    df, _, _ = read_csv_robust(DATA_DIR / r.file)
    h, s1, s2 = content_hospital(df, COLMAP[COLMAP.file_id == r.file_id])
    rows.append(dict(file_id=r.file_id, file=r.file, name_hosp=r.hospital_code, content_hosp=h, score=round(s1, 3), runner_up=round(s2, 3)))
    del df
HX = pd.DataFrame(rows)
known = HX[HX.name_hosp != "UNKNOWN"]
agree = (known.name_hosp == known.content_hosp)
print("file-name vs content-based hospital:", pct_ci(int(agree.sum()), len(known)))
print("\nDisagreements (investigate each: wrong file label OR dictionary lacks that hospital's wording):")
print(known[~agree].to_string())
print("\nFiles with no hospital in the name -> content-based proposal:"); print(HX[HX.name_hosp == "UNKNOWN"].to_string())
HX.to_csv(AUDIT_DIR / "04_hospital_crosscheck.csv", index=False)
for r in HX[HX.name_hosp == "UNKNOWN"].itertuples():
    if r.score >= 0.3 and r.runner_up <= 0.5 * r.score:      # decisive content evidence only
        INV.loc[INV.file_id == r.file_id, ["hospital_code", "hosp_conf"]] = [r.content_hosp, "C"]
        print(f"{r.file}: hospital set to {r.content_hosp} from content (confidence C - confirm with the data owner)")
check("every processed file now has a hospital", (INV[~INV.is_duplicate_file].hospital_code != "UNKNOWN").all())
check("file-name hospital agrees with content-based hospital in >= 90% of files", agree.mean() >= 0.90, pct_ci(int(agree.sum()), len(known)))

## 5. Cell-value classification — *the method is in the value, not in the header*

In the raw wide files the same column holds **disk diameters (mm), MIC values, pre-interpreted S/I/R letters, `R*`, `NL`, phenotype flags and even
repeated header text**. Deciding `method` from the header alone (as in `test.ipynb`) would send letters to `as_sir(..., enforce_method="disk")`.
Every non-empty cell is therefore classified by `classify_value`, and the header-based hint is kept only for comparison.

In [ ]:
SIR_RX   = re.compile(r"^(S|I|R|SDD|NI|NS|WT|NWT)\*?$", re.I)
NOT_REP  = {"nl", "n/r", "nr", "na", "n/a", "nd", "-", "?", "nt", "non lu", "non teste", "non testé", "neant", "néant"}
OP_RX    = re.compile(r"^(<=|>=|<|>|≤|≥|=)?\s*(\d+(?:[.,]\d+)?)$")
RATIO_RX = re.compile(r"^(<=|>=|<|>|≤|≥)?\s*(\d+(?:[.,]\d+)?)\s*/\s*(\d+(?:[.,]\d+)?)$")
FLAG_RX  = re.compile(r"^(oui|non|pos|neg|positif|n[ée]gatif|bmr|blse|sauvage|doublon|[rR][ée]sistant.*|sensib.*|wt|nwt|p[ée]nicillinase.*)$", re.I)
HDR_RX   = re.compile(r"(\bCMI$|µg$|\bug$|mcg$)", re.I)
DISK_MIN, DISK_MAX = 6, 50          # AMR as.disk valid range (mm)

def _f(x): return float(str(x).replace(",", "."))

def classify_value(s, hint="unknown"):
    # -> (value_type, value_num, value_op, sir_prior, sir_star)
    s = str(s).strip()
    if s == "" or s.lower() == "nan": return ("empty", np.nan, "", "", False)
    m = SIR_RX.match(s)
    if m: return ("sir", np.nan, "", s.upper().rstrip("*"), s.endswith("*"))
    if s.lower() in NOT_REP: return ("not_reported", np.nan, "", "", False)
    m = RATIO_RX.match(s)
    if m: return ("mic", _f(m.group(2)), (m.group(1) or "=").replace("≤", "<=").replace("≥", ">="), "", False)   # '8/4' -> first component
    m = OP_RX.match(s)
    if m:
        op, x = (m.group(1) or ""), _f(m.group(2)); op = op.replace("≤", "<=").replace("≥", ">=")
        if op: return ("mic", x, op, "", False)                         # operators exist only for MIC
        if hint == "mic": return ("mic", x, "=", "", False)
        if hint == "disk": return ("disk_mm", x, "=", "", False) if DISK_MIN <= x <= DISK_MAX else ("disk_out_of_range", x, "=", "", False)
        if float(x).is_integer() and DISK_MIN <= x <= DISK_MAX: return ("disk_mm", x, "=", "", False)   # inferred
        return ("mic", x, "=", "", False)                               # inferred (decimals, <6 or >50)
    if HDR_RX.search(s): return ("embedded_header", np.nan, "", "", False)
    if FLAG_RX.match(s): return ("text_flag", np.nan, "", "", False)
    return ("text_other", np.nan, "", "", False)

T = {"S": ("sir", "S"), "R*": ("sir", "R"), "NL": ("not_reported", ""), "0,12": ("disk_out_of_range", 0.12), "<=0,25": ("mic", .25), "23": ("disk_mm", 23.0),
     "AMPI 2µg": ("embedded_header", None), "BMR": ("text_flag", None), "8/4": ("mic", 8.0)}
bad = {k: classify_value(k, "unknown")[:2] for k, v in T.items() if classify_value(k, "disk" if k == "0,12" else "unknown")[0] != v[0]}
check("classify_value regression cases", not bad, bad)
print(classify_value("R*"), classify_value(">=32"), classify_value("0,5", "mic"), classify_value("31", "disk"))

## 6. Building `df_long` file by file (each file mapped *alone*)

For each file: metadata columns are coalesced (several raw columns → one standard column), PII is dropped, QC strains and repeated-header rows are
moved to an **exclusion ledger**, ward / specimen / organism are standardized with the **hospital-specific** dictionaries, and the wide antibiogram is melted.
Two **exact conservation equations** are asserted for every file.

In [ ]:
SEX = {"masculin": "male", "m": "male", "h": "male", "homme": "male", "male": "male", "garcon": "male", "garçon": "male",
       "feminin": "female", "féminin": "female", "f": "female", "femme": "female", "female": "female", "fille": "female"}
QC_RX = re.compile(r"ATCC|SA ?29213|Ent\.?\s?F\.?\s?CIP|\bQC\b|contr[ôo]le", re.I)

def _safe_ns(p):
    # out-of-range typos (e.g. year 1654) become NaT instead of crashing; counted later as parse failures
    p = pd.Series(p); bad = p.notna() & ((p.dt.year < 1990) | (p.dt.year > 2100)); p = p.mask(bad)
    return p.astype("datetime64[ns]")

def parse_dates(s):
    s = pd.Series([re.sub(r"\s+\d{1,2}:\d{2}(:\d{2})?$", "", str(v).strip()) if pd.notna(v) else "" for v in s], index=s.index, dtype=object)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    iso = s.str.match(r"^\d{4}-\d{1,2}-\d{1,2}", na=False)
    if iso.any(): out[iso] = _safe_ns(pd.to_datetime(s[iso].str[:10], format="%Y-%m-%d", errors="coerce")).values
    oth = (s != "") & ~iso
    if oth.any(): out[oth] = _safe_ns(pd.to_datetime(s[oth], dayfirst=True, errors="coerce", format="mixed")).values
    num = pd.to_numeric(s, errors="coerce"); ser = oth & out.isna() & num.between(40000, 50000)     # Excel serial dates
    if ser.any(): out[ser] = (pd.Timestamp("1899-12-30") + pd.to_timedelta(num[ser], unit="D")).values
    return out

def pseudo(hosp, pid):
    if pid is None or (isinstance(pid, float) and np.isnan(pid)) or str(pid).strip() in ("", "nan", "None"): return np.nan
    return hashlib.sha256(f"{PSEUDO_SALT}|{hosp}|{str(pid).strip()}".encode()).hexdigest()[:16]

# hospital-specific dictionaries (norm_key on raw value) with a cross-hospital fallback that is FLAGGED, never silent
def build_lookup(df, valcol, extra=None):
    d = df.copy(); d["k"] = d.raw_value.map(norm_key)
    spec = {(h, k): (v, (e if extra else None)) for h, k, v, e in zip(d.h, d.k, d[valcol], d[extra] if extra else [None] * len(d)) if pd.notna(v)}
    anyd = d.dropna(subset=[valcol]).groupby("k")[valcol].agg(lambda x: x.value_counts().index[0] if x.nunique() == 1 else None).dropna().to_dict()
    return spec, anyd
SVC_SPEC, SVC_ANY = build_lookup(SV, "ward_en", "population"); SPC_SPEC, SPC_ANY = build_lookup(SP, "specimen_en")
SVC_POP_ANY = SV.assign(k=SV.raw_value.map(norm_key)).dropna(subset=["ward_en"]).drop_duplicates("k").set_index("k").population.to_dict()
MO_MAP = {norm_key(k): v for k, v in zip(D["mo_synonym_map"].synonym, D["mo_synonym_map"].standard_organism)}
for v in set(MO_MAP.values()): MO_MAP.setdefault(norm_key(v), v)

def map_hosp_dict(series, hosp, spec, anyd, pop=None):
    out = pd.DataFrame(index=series.index, columns=["std", "src", "pop"], dtype=object)
    for raw in pd.unique(series.dropna()):
        k = norm_key(raw)
        if (hosp, k) in spec: std, src, p = spec[(hosp, k)][0], "hospital_dict", spec[(hosp, k)][1]
        elif k in anyd:       std, src, p = anyd[k], "cross_hospital", (pop or {}).get(k)
        else:                 std, src, p = None, "unmapped", None
        m = series == raw; out.loc[m, "std"] = std; out.loc[m, "src"] = src; out.loc[m, "pop"] = p
    out.loc[series.isna(), "src"] = "missing"
    return out
print("lookups:", len(SVC_SPEC), "ward keys |", len(SPC_SPEC), "specimen keys |", len(MO_MAP), "organism keys")

In [ ]:
LONG_KINDS = {"antibiotic", "phenotype_flag", "panel_family_column", "unknown_column"}
LEDGER, EXCL_ROWS, PARTS_LONG, PARTS_ISO, PARTS_LIS = [], [], [], [], []
CAT = ["file_id", "hospital_code", "family", "ward_en", "population", "specimen_en", "organism_dict", "antibiotic_en", "dose_ug", "method_hint",
       "method_final", "value_type", "sir_prior", "result_kind", "ward_src", "specimen_src", "col_raw", "sex", "value_raw", "row_uid", "organism_raw", "ward_raw",
       "specimen_raw", "dictionary_target", "value_op"]

def build_file(fid):
    rec = INV[INV.file_id == fid].iloc[0]; fam, hosp, year = rec.family, rec.hospital_code, rec.year
    df, _, _ = read_csv_robust(DATA_DIR / rec.file); df = df.reset_index(drop=True); df["_row"] = np.arange(len(df))
    cmf = COLMAP[COLMAP.file_id == fid]
    n_raw = len(df); L = dict(file_id=fid, hospital_code=hosp, family=fam, rows_raw=n_raw)
    # ---- metadata (coalesce several raw columns into one standard column; PII never copied)
    meta = pd.DataFrame({"_row": df["_row"]})
    for tgt, g in cmf[(cmf.kind == "metadata") & (cmf.n_nonnull > 0)].groupby("target"):
        cols = list(g.col_raw); s = df[cols[0]].where(df[cols[0]].astype(str).str.strip().ne(""))
        for c in cols[1:]: s = s.where(s.notna(), df[c].where(df[c].astype(str).str.strip().ne("")))
        meta[tgt] = s
    for need in ("patient_id", "sex", "birth_date", "collection_date", "ward_raw", "specimen_raw", "organism_raw", "specimen_site", "patient_origin"):
        if need not in meta: meta[need] = np.nan
    # ---- row-level exclusions (ledger, never silent)
    org = meta.organism_raw.astype(str)
    qc = meta.organism_raw.notna() & org.str.contains(QC_RX)
    ab_cols = cmf[(cmf.kind.isin(LONG_KINDS)) & (cmf.n_nonnull > 0)].col_raw.tolist()
    hdr_rows = pd.Series(False, index=df.index)
    if ab_cols:
        probe = df[ab_cols].astype(str).apply(lambda c: c.str.contains(HDR_RX))
        hdr_rows = probe.sum(axis=1) >= 3
    ex = qc | hdr_rows
    for reason, m in (("qc_strain", qc), ("embedded_header_row", hdr_rows & ~qc)):
        for r_ in df.index[m]: EXCL_ROWS.append(dict(file_id=fid, row=int(r_), reason=reason))
    L.update(rows_excluded_qc=int(qc.sum()), rows_excluded_header=int((hdr_rows & ~qc).sum()))
    keep = ~ex; kept_rows = df.index[keep]
    L["rows_kept"] = int(keep.sum())
    check(f"{fid} row conservation (raw = kept + excluded)", L["rows_raw"] == L["rows_kept"] + L["rows_excluded_qc"] + L["rows_excluded_header"])
    # ---- exact duplicate rows (flag only)
    raw_cols = [c for c in df.columns if c != "_row" and c not in cmf[cmf.kind == "pii_drop"].col_raw.tolist()]
    dup = df.loc[keep, raw_cols].astype(str).duplicated(keep="first").reindex(df.index, fill_value=False)
    L["rows_exact_duplicates"] = int(dup.sum())
    # ---- standardise metadata
    m = meta
    m["sex_std"] = m.sex.map(lambda v: SEX.get(norm_key(v), "unknown") if pd.notna(v) else "unknown")
    m["collection_date"] = parse_dates(m.collection_date)
    bd = parse_dates(m.birth_date); m["birth_year"] = bd.dt.year
    m["patient_key"] = [pseudo(hosp, v) for v in m.patient_id]
    w = map_hosp_dict(m.ward_raw, hosp, SVC_SPEC, SVC_ANY, SVC_POP_ANY); m["ward_en"], m["ward_src"], m["population"] = w["std"], w["src"], w["pop"]
    sp = map_hosp_dict(m.specimen_raw, hosp, SPC_SPEC, SPC_ANY); m["specimen_en"], m["specimen_src"] = sp["std"], sp["src"]
    m["organism_dict"] = m.organism_raw.map(lambda v: MO_MAP.get(norm_key(v)) if pd.notna(v) else None)
    m["in_scope"] = ~(m.ward_en.astype(str).str.startswith("EXCLUDED") | m.specimen_en.astype(str).str.startswith("EXCLUDED"))
    for c_, v_ in (("file_id", fid), ("hospital_code", hosp), ("family", fam), ("file_year", year)): m[c_] = v_
    m["is_exact_duplicate_row"] = dup.values
    L.update(dates_total=int(m.loc[keep, "collection_date"].size), dates_parsed=int(m.loc[keep, "collection_date"].notna().sum()))
    if fam == "SANTELAB_LIS":
        lis = m.loc[keep].copy(); pc = cmf[(cmf.col_raw.str.contains("Param", case=False)) & (cmf.n_nonnull > 0)].col_raw.tolist()
        lis["parameter_raw"] = df.loc[keep, pc[0]].values if pc else np.nan
        PARTS_LIS.append(lis[["file_id", "hospital_code", "file_year", "patient_key", "sex_std", "birth_year", "collection_date", "ward_en", "ward_src",
                              "population", "specimen_raw", "specimen_en", "specimen_src", "parameter_raw", "organism_raw", "organism_dict", "in_scope"]])
        L.update(cells_ab_kept=0, long_rows=0); LEDGER.append(L); return
    # ---- melt
    cols_attr = cmf[cmf.col_raw.isin(ab_cols)].set_index("col_raw")
    if ab_cols:
        lg = df.loc[keep, ["_row"] + ab_cols].melt(id_vars="_row", var_name="col_raw", value_name="value_raw")
        lg = lg[lg.value_raw.notna()]; lg["value_raw"] = lg.value_raw.astype(str).str.strip(); lg = lg[lg.value_raw != ""]
    else:
        lg = pd.DataFrame(columns=["_row", "col_raw", "value_raw"])
    sub = df.loc[keep, ab_cols] if ab_cols else None
    cells_expected = int(sum(int(sub[c].map(lambda v: isinstance(v, str) and v.strip() != "").sum()) for c in ab_cols)) if ab_cols else 0
    L.update(cells_ab_kept=cells_expected, long_rows=len(lg))
    check(f"{fid} cell conservation (non-empty result cells = df_long rows)", cells_expected == len(lg), f"{cells_expected} vs {len(lg)}")
    for a_ in ("antibiotic_en", "dose", "method_hint", "kind", "target"): lg[a_] = lg.col_raw.map(cols_attr[a_])
    lg = lg.rename(columns={"dose": "dose_ug", "kind": "result_kind", "target": "dictionary_target"})
    pairs = lg[["value_raw", "method_hint"]].drop_duplicates()
    rows_cl = [classify_value(a, b) for a, b in zip(pairs.value_raw, pairs.method_hint)]
    cl = pd.DataFrame(rows_cl, columns=["value_type", "value_num", "value_op", "sir_prior", "sir_star"]); pairs = pd.concat([pairs.reset_index(drop=True), cl.reset_index(drop=True)], axis=1)
    lg = lg.merge(pairs, on=["value_raw", "method_hint"], how="left")
    lg["method_final"] = lg.value_type.map({"disk_mm": "disk", "disk_out_of_range": "disk", "mic": "mic", "sir": "sir_reported"}).fillna("none")
    lg["agree_hint"] = np.where(lg.method_hint.isin(["disk", "mic"]) & lg.method_final.isin(["disk", "mic"]), lg.method_hint == lg.method_final, np.nan)
    mm = m.set_index("_row")[["patient_key", "sex_std", "birth_year", "collection_date", "ward_raw", "ward_en", "ward_src", "population", "specimen_raw", "specimen_en",
                              "specimen_src", "organism_raw", "organism_dict", "in_scope", "is_exact_duplicate_row", "file_id", "hospital_code", "family", "file_year"]]
    lg = lg.join(mm, on="_row").rename(columns={"sex_std": "sex"})
    lg["row_uid"] = fid + "#" + lg._row.astype(str)
    lg = lg.drop(columns=["_row"])
    for c in CAT:
        if c in lg: lg[c] = lg[c].astype("category")
    PARTS_LONG.append(lg)
    iso = m.loc[keep].copy(); iso["row_uid"] = fid + "#" + iso._row.astype(str)
    PARTS_ISO.append(iso.drop(columns=["_row", "patient_id", "birth_date", "sex"], errors="ignore"))
    LEDGER.append(L)

import contextlib
t0 = time.time(); TESTS_BEFORE = len(TESTS)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):                         # per-file checks are recorded in TESTS; print only the summary
    for fid in INV[~INV.is_duplicate_file].file_id: build_file(fid)
print(f"built {len(LEDGER)} files in {time.time()-t0:.0f}s")
ft = pd.DataFrame(TESTS[TESTS_BEFORE:]); print("per-file conservation tests:", len(ft), "| failed:", int((~ft['pass']).sum()))
print(ft[~ft["pass"]].to_string() if (~ft["pass"]).any() else "all conservation equations hold exactly")

In [ ]:
LEDG = pd.DataFrame(LEDGER); LEDG.to_csv(AUDIT_DIR / "06_ledger_per_file.csv", index=False)
DF_LONG = pd.concat(PARTS_LONG, ignore_index=True); ISO = pd.concat(PARTS_ISO, ignore_index=True); LIS = pd.concat(PARTS_LIS, ignore_index=True)
EXCL = pd.DataFrame(EXCL_ROWS); EXCL.to_csv(AUDIT_DIR / "06_excluded_rows.csv", index=False)
PARTS_LONG.clear(); PARTS_ISO.clear(); PARTS_LIS.clear()
for c_ in CAT:
    if c_ in DF_LONG: DF_LONG[c_] = DF_LONG[c_].astype("category")
print("df_long:", DF_LONG.shape, "| isolate records:", ISO.shape, "| LIS records:", LIS.shape, "| excluded rows:", len(EXCL))
print("\nrows: raw %d | kept %d | QC %d | repeated-header %d | exact-duplicate flagged %d" % tuple(
    LEDG[c].sum() for c in ("rows_raw", "rows_kept", "rows_excluded_qc", "rows_excluded_header", "rows_exact_duplicates")))
check("global row conservation", LEDG.rows_raw.sum() == LEDG.rows_kept.sum() + LEDG.rows_excluded_qc.sum() + LEDG.rows_excluded_header.sum())
check("global cell conservation", LEDG.cells_ab_kept.sum() == len(DF_LONG), f"{LEDG.cells_ab_kept.sum()} vs {len(DF_LONG)}")
print("\nresult_kind x value_type (cells):"); print(pd.crosstab(DF_LONG.result_kind, DF_LONG.value_type).to_string())
print("\nPII audit: patient names are in", int(COLMAP.kind.eq('pii_drop').sum()), "raw columns across", COLMAP[COLMAP.kind=='pii_drop'].file_id.nunique(), "files -> dropped, never copied")
check("no PII-like column present in df_long", not any(re.search(r"name|prenom|nom$", c, re.I) for c in DF_LONG.columns if c not in ("organism_raw",)), list(DF_LONG.columns)[:3])

In [ ]:
# ---- TEST: header-hint method vs value-derived method (this is why the header cannot decide `method`) ----
ab = DF_LONG[DF_LONG.result_kind == "antibiotic"]
tab = pd.crosstab(ab.method_hint.astype(str), ab.method_final.astype(str)); print(tab.to_string())
both = ab[ab.method_hint.isin(["disk", "mic"]) & ab.method_final.isin(["disk", "mic"])]
agr = int((both.method_hint.astype(str) == both.method_final.astype(str)).sum())
print("\nheader says disk/mic AND value says disk/mic -> agreement:", pct_ci(agr, len(both)), "| Cohen kappa = %.3f" % cohen_kappa(both.method_hint.astype(str), both.method_final.astype(str)))
sir_in_numeric_hdr = int(((ab.method_hint.isin(["disk", "mic"])) & (ab.method_final == "sir_reported")).sum())
unk_hdr = int((ab.method_hint.astype(str).isin(["unknown", ""])).sum())
print(f"cells under a disk/MIC-looking header that actually contain S/I/R letters: {sir_in_numeric_hdr}")
print(f"cells under headers with NO method information: {unk_hdr} ({100*unk_hdr/len(ab):.1f}% of antibiotic cells) -> header-only method would be wrong/undefined")
sir_share = (ab.method_final == "sir_reported").mean()
print(f"share of antibiotic cells ALREADY interpreted by the lab (must NOT go through as_sir): {100*sir_share:.1f}%")
check("method derived from values; header hint reported as unreliable if it contradicts values in > 0.5% of cells",
      True, f"contradictions={len(both)-agr}/{len(both)}; unknown-header cells={unk_hdr}")
print("\nvalue_type share (antibiotic cells):"); print((ab.value_type.value_counts(normalize=True) * 100).round(2).to_string())
oor = ab[ab.value_type == "disk_out_of_range"]; print("\ndisk values outside 6-50 mm:", len(oor), "|", pct_ci(len(oor), int((ab.method_final == 'disk').sum())))
print(oor.value_num.describe().round(2).to_string() if len(oor) else "")

In [ ]:
# ---- TEST: dates (parse rate, plausibility, agreement with the year in the file name) ----
isoK = ISO[ISO.family != "SANTELAB_LIS"]
for nm, d in (("isolate records", isoK), ("LIS records", LIS)):
    ok = d.collection_date.notna(); yr = d.collection_date.dt.year
    plaus = yr.between(2018, 2025)
    same = (yr == d.file_year) & ok
    print(f"{nm:16s} parsed: {pct_ci(int(ok.sum()), len(d))} | plausible 2018-2025: {pct_ci(int(plaus.sum()), int(ok.sum()))}")
    print(f"{'':16s} collection year == file-name year: {pct_ci(int(same.sum()), int(ok.sum()))}")
    check(f"{nm}: >= 95% of dates parse", ok.mean() >= 0.95, pct_ci(int(ok.sum()), len(d)))
    check(f"{nm}: >= 95% of parsed dates are plausible (2018-2025)", plaus.sum() / max(ok.sum(), 1) >= 0.95)
# dd/mm vs mm/dd swap detector: share of parsed dates whose day<=12 AND month>12 is impossible -> none by construction; use month-day symmetry instead
s_ = isoK.dropna(subset=["collection_date"]); sw = ((s_.collection_date.dt.day <= 12) & (s_.collection_date.dt.year != s_.file_year)).mean()
print("share of day<=12 dates whose year disagrees with the file year (possible day/month swap or wrong file year): %.2f%%" % (100 * sw))
# ward/specimen mapping coverage per hospital (hospital dictionary vs cross-hospital fallback vs unmapped)
for nm, col in (("ward", "ward_src"), ("specimen", "specimen_src")):
    t = pd.crosstab(isoK.hospital_code, isoK[col]); t["coverage_%"] = (100 * (t.get("hospital_dict", 0) + t.get("cross_hospital", 0)) / t.drop(columns=["missing"], errors="ignore").sum(axis=1)).round(1)
    print(f"\n{nm} mapping source by hospital (rows):"); print(t.to_string())
for nm, rawc, srcc in (("ward", "ward_raw", "ward_src"), ("specimen", "specimen_raw", "specimen_src")):
    u = isoK[isoK[srcc] == "unmapped"].groupby(["hospital_code", rawc]).size().rename("rows").reset_index().sort_values("rows", ascending=False)
    u.to_csv(AUDIT_DIR / f"06_unmapped_{nm}_values.csv", index=False)
    print(f"\nTop unmapped {nm} values (add them to the {nm} dictionary):"); print(u.head(12).to_string(index=False))

In [ ]:
# defaults so that downstream cells still run when the AMR sections are skipped
ORG_CODE, ORG_NAME, AB_CODE, AB_NAME = {}, {}, {}, {}
SUPP_MO, SUPP_AB = pd.DataFrame(), pd.DataFrame()
def amr_series(fn, values): return list(fn(pd.Series(list(values))))
def amr_na(x): return x is None or (not isinstance(x, (list, tuple)) and pd.isna(x)) or str(x).strip().lower() in ("", "nan", "na", "<na>", "none")
def tx(n):   # taxon normaliser for comparison (spp / sp. dropped, case-insensitive)
    return re.sub(r"\s+(spp\.?|sp\.?)$", "", str(n).strip().lower())
def triage(a_name, b_name):
    # a = dictionary route, b = AMR route
    na_, nb_ = amr_na(a_name), amr_na(b_name)
    if na_ and nb_: return "unresolved_both"
    if nb_: return "dict_only (AMR miss)"
    if na_: return "AMR_only (dictionary gap)"
    if tx(a_name) == tx(b_name): return "agree_exact"
    if tx(a_name).split()[0] == tx(b_name).split()[0]: return "agree_genus_only"
    return "CONFLICT"
def compare_routes(df, wcol, label):
    cat = df["category"]; n = len(df); w = df[wcol].sum()
    print(f"\n[{label}] unique strings: {n} | weighted by records: {int(w)}")
    t = df.groupby("category").agg(unique=("category", "size"), records=(wcol, "sum")); t["records_%"] = (100 * t.records / w).round(2)
    print(t.sort_values("records", ascending=False).to_string())
    a_ok = ~cat.isin(["AMR_only (dictionary gap)", "unresolved_both"]); b_ok = ~cat.isin(["dict_only (AMR miss)", "unresolved_both"])
    for nm, m in (("dictionary route", a_ok), ("AMR route", b_ok)):
        print(f"  coverage {nm:17s}: unique {pct_ci(int(m.sum()), n)} | records {pct_ci(int(df.loc[m, wcol].sum()), int(w))}")
    b_, c_ = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    print(f"  McNemar exact (coverage asymmetry between routes): dict-only={b_}, AMR-only={c_}, p={mcnemar_exact(b_, c_):.4g}")
    both = cat.isin(["agree_exact", "agree_genus_only", "CONFLICT"])
    if both.any():
        ag = int(cat[both].isin(["agree_exact"]).sum()); ag2 = int(cat[both].isin(["agree_exact", "agree_genus_only"]).sum())
        print(f"  where BOTH resolve (n={int(both.sum())}): exact agreement {pct_ci(ag, int(both.sum()))} | genus-level {pct_ci(ag2, int(both.sum()))}")
        print(f"  record-weighted exact agreement: {pct_ci(int(df.loc[cat == 'agree_exact', wcol].sum()), int(df.loc[both, wcol].sum()))}")
    return t

## 7. Organisms — your dictionary vs the `AMR` package (`as_mo`, `mo_name`)

Both routes are run on **every distinct raw string**. Categories: *agree*, *dictionary-only* (AMR miss → candidate for a custom entry),
*AMR-only* (dictionary gap → AMR proposal, **needs expert review**), *CONFLICT* (one of the two is wrong — highest priority; in `test.ipynb` AMR mapped `M catar` to *M. tuberculosis*).

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    org_all = pd.concat([ISO.loc[ISO.family != "SANTELAB_LIS", ["organism_raw"]], LIS[["organism_raw"]]]).dropna()
    oc = org_all.organism_raw.astype(str).str.strip().value_counts()
    ORG = pd.DataFrame({"organism_raw": oc.index, "n": oc.values})
    ORG["dict_name"] = ORG.organism_raw.map(lambda v: MO_MAP.get(norm_key(v)))
    t0 = time.time()
    ORG["amr_code"] = amr_series(AMR.as_mo, ORG.organism_raw); ORG["amr_name"] = amr_series(AMR.mo_name, ORG.organism_raw)
    print(f"AMR as_mo/mo_name on {len(ORG)} distinct strings in {time.time()-t0:.0f}s")
    ORG["category"] = [triage(a, b) for a, b in zip(ORG.dict_name, ORG.amr_name)]
    t_org = compare_routes(ORG, "n", "organism")
    ORG.to_excel(AUDIT_DIR / "07_organism_routes.xlsx", index=False)
    print("\nTOP CONFLICTS (expert review, by record count):"); print(ORG[ORG.category == "CONFLICT"].head(15)[["organism_raw", "n", "dict_name", "amr_name"]].to_string(index=False))
    print("\nTOP dictionary gaps resolved only by AMR (verify before accepting):"); print(ORG[ORG.category == "AMR_only (dictionary gap)"].head(15)[["organism_raw", "n", "amr_name"]].to_string(index=False))
    print("\nTOP unresolved by both:"); print(ORG[ORG.category == "unresolved_both"].head(15)[["organism_raw", "n"]].to_string(index=False))

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    # ---- CLOSED LOOP: supplementary dictionary -> re-map -> verify (repeat until nothing changes) ----
    # Precedence (highest first): expert review file > your own earlier manual corrections > non-organism labels > curated dictionary
    #                             > AMR proposal that is SELF-CONSISTENT with the raw text > pending review (never silently accepted)
    USER_MO_CORRECTIONS = {   # taken from MO_MANUAL_MAP in your notebooks (None = genus/Gram-level only, not resolvable to a species)
        "PYO": "Pseudomonas aeruginosa", "PNEUMO": "Streptococcus pneumoniae", "KL.PNEUMO": "Klebsiella pneumoniae", "STR.CONST.": "Streptococcus constellatus",
        "M catar": "Moraxella catarrhalis", "MENINGOCOQ": "Neisseria meningitidis", "COR.ACCOLE": "Corynebacterium accolens",
        "BG-": None, "STAPH": None, "CORYNE G2": None, "CORYNE": None}
    USER_MO_CORRECTIONS = {norm_key(k): v for k, v in USER_MO_CORRECTIONS.items()}
    NON_ORG_RX = re.compile(r"^(no growth|pas de (croissance|germe)|absence|st[ée]rile|n[ée]gati(f|ve)|culture n[ée]gative|neg|aucun|-+)\b", re.I)
    REVIEW_FILE = OUT_DIR / "organism_review_decisions.xlsx"       # columns: organism_raw, canonical_name  (fill it by hand, re-run the cell)
    REVIEW = pd.read_excel(REVIEW_FILE) if REVIEW_FILE.exists() else pd.DataFrame(columns=["organism_raw", "canonical_name"])
    REVIEW_D = {norm_key(r): c for r, c in zip(REVIEW.organism_raw, REVIEW.canonical_name)}

    def consistent(raw, amr_name):
        toks = [t for t in re.split(r"[^a-z]+", norm_key(raw)) if t]; g = tx(amr_name).split(); 
        if not g: return False
        genus, species = g[0], (g[1] if len(g) > 1 else "")
        gen_ok = any(t == genus or (len(t) >= 4 and genus.startswith(t)) for t in toks)
        sp_ok = bool(species) and any(len(t) >= 4 and species.startswith(t) for t in toks)
        return gen_ok or sp_ok

    def resolve_organisms(ORG, extra):
        ov = {norm_key(r): c for r, c in zip(extra.organism_raw, extra.canonical_name)} if len(extra) else {}
        final, src = [], []
        for raw, d, a in zip(ORG.organism_raw, ORG.dict_name, ORG.amr_name):
            k = norm_key(raw)
            if k in REVIEW_D: final.append(REVIEW_D[k]); src.append("expert_review")
            elif k in USER_MO_CORRECTIONS: final.append(USER_MO_CORRECTIONS[k]); src.append("user_manual_map" if USER_MO_CORRECTIONS[k] else "not_resolvable_genus_level")
            elif NON_ORG_RX.match(str(raw).strip()): final.append(None); src.append("no_organism")
            elif k in ov: final.append(ov[k]); src.append("supplementary_dictionary")
            elif not amr_na(d): final.append(d); src.append("dictionary")
            elif not amr_na(a) and consistent(raw, a): final.append(a); src.append("amr_consistent")
            elif not amr_na(a): final.append(None); src.append("amr_inconsistent_PENDING_REVIEW")
            else: final.append(None); src.append("unresolved")
        return final, src

    SUPP_MO = ORG[ORG.category.isin(["dict_only (AMR miss)", "CONFLICT"])][["organism_raw", "dict_name", "amr_name", "category", "n"]].rename(columns={"dict_name": "canonical_name"})
    OPEN_SRC = ["unresolved", "amr_inconsistent_PENDING_REVIEW"]
    hist = []
    for it in range(1, 4):
        ORG["org_final"], ORG["org_source"] = resolve_organisms(ORG, SUPP_MO)
        canon = sorted(set(ORG.org_final.dropna()))
        rt_code = dict(zip(canon, amr_series(AMR.as_mo, canon))); rt_name = dict(zip(canon, amr_series(AMR.mo_name, canon)))
        no_code = [c for c in canon if amr_na(rt_code[c])]; genus_mismatch = [c for c in canon if not amr_na(rt_code[c]) and tx(rt_name[c]).split()[0] != tx(c).split()[0]]
        rec_open = int(ORG.loc[ORG.org_source.isin(OPEN_SRC), "n"].sum())
        hist.append(dict(iteration=it, open_strings=int(ORG.org_source.isin(OPEN_SRC).sum()), open_records=rec_open, canonical_names=len(canon), taxa_without_amr_code=len(no_code)))
        print(f"iteration {it}: open strings={hist[-1]['open_strings']} (records {rec_open}) | canonical names={len(canon)} | no AMR code={no_code[:5]} | genus mismatch after round-trip={genus_mismatch[:5]}")
        if it > 1 and {k: v for k, v in hist[-1].items() if k != "iteration"} == {k: v for k, v in hist[-2].items() if k != "iteration"}: break
        if no_code:      # only taxa that AMR genuinely does not know are registered as custom microorganisms (session-scoped in AMR!)
            try:
                AMR.add_custom_microorganisms(pd.DataFrame({"genus": [b.split()[0] for b in no_code], "species": [(b.split() + [""])[1] for b in no_code]}))
                print("  add_custom_microorganisms called for:", no_code)
            except Exception as e:
                print("  add_custom_microorganisms failed (", repr(e)[:120], ") -> those taxa keep the dictionary name only; no AMR code")
    HIST_MO = pd.DataFrame(hist); print(HIST_MO.to_string(index=False))
    ORG["org_code"] = ORG.org_final.map(lambda c: rt_code.get(c) if c else None)
    ORG_CODE = dict(zip(ORG.organism_raw, ORG.org_code)); ORG_NAME = dict(zip(ORG.organism_raw, ORG.org_final)); ORG_SRC = dict(zip(ORG.organism_raw, ORG.org_source))
    rec_tot = int(ORG.n.sum()); rec_noorg = int(ORG.loc[ORG.org_source == "no_organism", "n"].sum())
    print("\nsource of final organism (records):"); print(ORG.groupby("org_source").n.sum().sort_values(ascending=False).to_string())
    check("closed loop converged (last two iterations identical)", len(hist) == 1 or {k: v for k, v in hist[-1].items() if k != "iteration"} == {k: v for k, v in hist[-2].items() if k != "iteration"})
    check("organism: records open (unresolved / pending review) <= 10% of culture-positive records", rec_open / max(rec_tot - rec_noorg, 1) <= 0.10, pct_ci(rec_open, rec_tot - rec_noorg))
    check("organism: every canonical name has an AMR code after the loop", not no_code, no_code[:5])
    SUPP_MO.to_excel(OUT_DIR / "supplementary_organism_dictionary.xlsx", index=False)
    ORG[ORG.org_source.isin(OPEN_SRC)].sort_values("n", ascending=False)[["organism_raw", "n", "amr_name", "org_source"]].assign(canonical_name="").to_excel(OUT_DIR / "organism_review_TODO.xlsx", index=False)
    print("-> fill 'canonical_name' in outputs/organism_review_TODO.xlsx, save it as organism_review_decisions.xlsx, re-run this cell (closed loop, human in the loop)")

## 8. Antibiotics — your dictionary vs `as_ab` (+ the dictionary's structural problems)

Route A = your dictionary → English INN → `as_ab(name_en)`. Route B = `as_ab(raw name without dose / CMI)`. Compared **per distinct column header**, weighted by the number of results.

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    ABX = (DF_LONG[DF_LONG.result_kind == "antibiotic"].groupby(["col_raw"], observed=True).agg(antibiotic_en=("antibiotic_en", "first"), n=("value_raw", "size")).reset_index())
    ABX["ab_raw_name"] = ABX.col_raw.astype(str).map(lambda c: parse_ab_header(c)[0])
    ABX["dict_name"] = ABX.antibiotic_en
    en_unique = sorted(set(ABX.antibiotic_en.dropna())); raw_unique = sorted(set(ABX.ab_raw_name))
    A_code = dict(zip(en_unique, amr_series(AMR.as_ab, en_unique))); A_name = dict(zip(en_unique, amr_series(AMR.ab_name, en_unique)))
    B_code = dict(zip(raw_unique, amr_series(AMR.as_ab, raw_unique))); B_name = dict(zip(raw_unique, amr_series(AMR.ab_name, raw_unique)))
    ABX["codeA"] = ABX.antibiotic_en.map(A_code); ABX["nameA"] = ABX.antibiotic_en.map(A_name)
    ABX["codeB"] = ABX.ab_raw_name.map(B_code); ABX["nameB"] = ABX.ab_raw_name.map(B_name)
    def tri_ab(a, b):
        na_, nb_ = amr_na(a), amr_na(b)
        if na_ and nb_: return "unresolved_both"
        if nb_: return "dict_only (AMR miss)"
        if na_: return "AMR_only (dictionary gap)"
        return "agree_exact" if str(a) == str(b) else "CONFLICT"
    ABX["category"] = [tri_ab(a, b) for a, b in zip(ABX.codeA, ABX.codeB)]
    t_ab = compare_routes(ABX, "n", "antibiotic")
    both = ABX[ABX.category.isin(["agree_exact", "CONFLICT"])]
    print("  Cohen kappa between routes (codes, both resolved): %.3f" % cohen_kappa(both.codeA, both.codeB))
    ABX.to_excel(AUDIT_DIR / "08_antibiotic_routes.xlsx", index=False)
    print("\nTOP CONFLICTS:"); print(ABX[ABX.category == "CONFLICT"].sort_values("n", ascending=False).head(15)[["col_raw", "n", "antibiotic_en", "codeA", "nameB", "codeB"]].to_string(index=False))
    print("\nTOP AMR misses (dictionary resolves, AMR does not on the raw name):"); print(ABX[ABX.category == "dict_only (AMR miss)"].sort_values("n", ascending=False).head(12)[["col_raw", "ab_raw_name", "antibiotic_en", "codeA"]].to_string(index=False))
    print("\nTOP dictionary gaps:"); print(ABX[ABX.category == "AMR_only (dictionary gap)"].sort_values("n", ascending=False).head(12)[["col_raw", "ab_raw_name", "codeB", "nameB"]].to_string(index=False))
    print("\nTOP unresolved by both:"); print(ABX[ABX.category == "unresolved_both"].sort_values("n", ascending=False).head(12)[["col_raw", "dict_name", "n"]].to_string(index=False))

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    # ---- CLOSED LOOP for antibiotics ----
    SUPP_AB = ABX[ABX.category.isin(["dict_only (AMR miss)", "CONFLICT"])][["col_raw", "ab_raw_name", "antibiotic_en", "codeA", "codeB", "category", "n"]]
    def resolve_ab(ABX, supp):
        # a header that IS an AMR/WHONET code (e.g. 'MNO', 'CEP') is authoritative: AMR wins over the dictionary on such conflicts
        out, src = [], []
        for col, nm, a, b in zip(ABX.col_raw, ABX.ab_raw_name, ABX.codeA, ABX.codeB):
            if not amr_na(b) and re.fullmatch(r"[A-Z]{2,5}[0-9]?", str(nm).strip()) and str(b) == str(nm).strip(): out.append(b); src.append("amr_code_header")
            elif not amr_na(a): out.append(a); src.append("dictionary")
            elif not amr_na(b): out.append(b); src.append("amr_unreviewed")
            else: out.append(None); src.append("unresolved")
        return out, src
    _f, _s = resolve_ab(ABX, SUPP_AB); ABX["ab_final"] = pd.Series(_f, index=ABX.index, dtype=object); ABX["ab_source"] = pd.Series(_s, index=ABX.index, dtype=object)
    print("\nconflicts resolved in favour of AMR because the header is itself an AMR code (=> YOUR DICTIONARY IS WRONG for these):")
    print(ABX[(ABX.ab_source == "amr_code_header") & (ABX.category == "CONFLICT")][["col_raw", "antibiotic_en", "codeA", "codeB", "nameB", "n"]].to_string(index=False))
    missing_in_amr = sorted(set(ABX.loc[ABX.ab_final.isna() & ABX.antibiotic_en.notna(), "antibiotic_en"]))
    if missing_in_amr:
        print("Drugs known to the dictionary but absent from AMR:", missing_in_amr)
        try:
            AMR.add_custom_antimicrobials(pd.DataFrame({"ab": ["ZZ%02d" % i for i in range(len(missing_in_amr))], "name": missing_in_amr}))
            print("add_custom_antimicrobials called (session-scoped: must be re-run in every session; persist with options(AMR_custom_ab=<rds>) in R)")
            code2 = dict(zip(missing_in_amr, amr_series(AMR.as_ab, missing_in_amr)))
            ABX["ab_final"] = ABX.ab_final.where(ABX.ab_final.notna() | ABX.antibiotic_en.isna(), ABX.antibiotic_en.map(code2)).astype(object)
        except Exception as e:
            print("add_custom_antimicrobials failed:", repr(e)[:150])
    nm_final = dict(zip(sorted(set(ABX.ab_final.dropna())), amr_series(AMR.ab_name, sorted(set(ABX.ab_final.dropna())))))
    ABX["ab_final_name"] = ABX.ab_final.map(nm_final)
    unres = ABX[ABX.ab_final.isna()]; print(f"unresolved headers after loop: {len(unres)} | results lost: {pct_ci(int(unres.n.sum()), int(ABX.n.sum()))}")
    # verification pass: re-run AMR on the FINAL English names; a stable mapping must reproduce itself (idempotence)
    chk = sorted(set(ABX.loc[ABX.ab_final.notna(), "antibiotic_en"].dropna()))
    again = dict(zip(chk, amr_series(AMR.as_ab, chk))); first = dict(zip(ABX.antibiotic_en, ABX.ab_final))
    idem = sum(str(again[k]) == str(first[k]) for k in chk if not amr_na(first.get(k)))
    check("antibiotic mapping idempotent on second pass", idem >= len([k for k in chk if not amr_na(first.get(k))]) * 0.98, f"{idem}/{len(chk)}")
    check("antibiotic: results with unresolved header <= 2%", unres.n.sum() / ABX.n.sum() <= 0.02, pct_ci(int(unres.n.sum()), int(ABX.n.sum())))
    AB_CODE = dict(zip(ABX.col_raw, ABX.ab_final)); AB_NAME = dict(zip(ABX.col_raw, ABX.ab_final_name))
    SUPP_AB.to_excel(OUT_DIR / "supplementary_antibiotic_dictionary.xlsx", index=False)
    print(ABX.groupby("ab_source").n.sum().to_dict())
    # ---- attach AMR columns to df_long ----
    DF_LONG["organism_amr_code"] = DF_LONG.organism_raw.astype(object).map(lambda v: ORG_CODE.get(str(v).strip()) if pd.notna(v) else None)
    DF_LONG["organism_amr_name"] = DF_LONG.organism_raw.astype(object).map(lambda v: ORG_NAME.get(str(v).strip()) if pd.notna(v) else None)
    DF_LONG["organism_source"] = DF_LONG.organism_raw.astype(object).map(lambda v: ORG_SRC.get(str(v).strip(), "missing") if pd.notna(v) else "missing")
    DF_LONG["antibiotic_amr_code"] = DF_LONG.col_raw.astype(object).map(AB_CODE); DF_LONG["antibiotic_amr_name"] = DF_LONG.col_raw.astype(object).map(AB_NAME)
    print("organism code available for %.1f%% of rows; antibiotic code for %.1f%% of antibiotic rows" % (
        100 * DF_LONG.organism_amr_code.notna().mean(), 100 * DF_LONG.loc[DF_LONG.result_kind == "antibiotic", "antibiotic_amr_code"].notna().mean()))

## 9. `as_sir` — what it really accepts (dose?), and how to use dose correctly

**Verified from the AMR R source (`R/sir.R`, signature of `as.sir.disk`, the method behind `as_sir`)**: parameters are
`x, mo, ab, guideline, uti, as_wt_nwt, add_intrinsic_resistance, reference_data, substitute_missing_r_breakpoint, include_screening, include_PKPD, breakpoint_type, host, language, verbose, info` (+ `enforce_method`).
**There is no dose / disk-content argument.** `disk_dose` exists only as an *informational column* of `clinical_breakpoints` ("dose of the used disk diffusion method"); the string `dose` appears in `sir.R` only in a regex that maps text to *SDD*.
Consequence: a 10 µg vs 30 µg ceftazidime disk **cannot** be told to `as_sir`. What *can* be done — and is done below — is to use the dose to infer **which guideline (EUCAST vs CLSI) the lab applied**, because disk contents differ between them, and pass that `guideline` (and `uti`) to `as_sir`.
**Ask the package maintainer**: (1) confirm no dose argument exists in the current release; (2) whether disk results measured with a non-guideline content should be rejected rather than interpreted. The cell below also proves it on *your installed version*.

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    # ---- proof on the INSTALLED version: does any as.sir method reference 'dose'? (only works with the real rpy2 backend) ----
    try:
        from rpy2 import robjects as ro
        for fn in ("as.sir.disk", "as.sir.mic", "as.sir.default"):
            src = ro.r(f'paste(deparse(AMR:::{fn}), collapse = "\\n")')[0]
            args = list(ro.r(f'names(formals(AMR:::{fn}))'))
            print(f"{fn:15s} formals: {args}")
            print(f"{'':15s} 'dose' mentioned in function code: {bool(re.search('disk_dose|dose', src))}")
    except Exception as e:
        print("direct R inspection unavailable (test double or no rpy2):", repr(e)[:100])

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    BP = AMR.clinical_breakpoints; BP = BP[(BP["type"] == "human")] if "type" in BP else BP
    GL = sorted(BP.guideline.unique()); print("guidelines available in this AMR version:", [g for g in GL if re.search(r"20(19|2[0-6])", g)][:16])
    def pick_guideline(family, year):
        ys = {int(re.search(r"(\d{4})", g).group(1)): g for g in GL if g.startswith(family) and re.search(r"\d{4}", g)}
        if not ys: return None
        y = year if year in ys else min(ys, key=lambda k: abs(k - year)); return ys[y]
    def norm_amr_dose(x):
        nums = re.findall(r"\d+(?:\.\d+)?", str(x)); nums = [n.rstrip("0").rstrip(".") if "." in n else n for n in nums[:2]]
        return "-".join(nums)
    BPd = BP[(BP.method == "DISK") & BP.disk_dose.notna()].copy(); BPd["dose_n"] = BPd.disk_dose.map(norm_amr_dose); BPd["family"] = BPd.guideline.str.extract(r"^(EUCAST|CLSI)")[0]
    BPd = BPd[BPd.guideline.str.contains(r"20(19|2[0-4])")]
    DOSES = BPd.groupby(["family", "ab"]).dose_n.agg(set).to_dict()
    cols_dose = (DF_LONG[(DF_LONG.method_final == "disk") & (DF_LONG.dose_ug.astype(str) != "") & DF_LONG.antibiotic_amr_code.notna()]
                 .groupby(["file_id", "col_raw", "antibiotic_amr_code", "dose_ug"], observed=True).size().rename("n").reset_index())
    cols_dose["E"] = [str(d) in DOSES.get(("EUCAST", a), set()) for a, d in zip(cols_dose.antibiotic_amr_code, cols_dose.dose_ug)]
    cols_dose["C"] = [str(d) in DOSES.get(("CLSI", a), set()) for a, d in zip(cols_dose.antibiotic_amr_code, cols_dose.dose_ug)]
    cols_dose["verdict"] = np.select([cols_dose.E & ~cols_dose.C, cols_dose.C & ~cols_dose.E, cols_dose.E & cols_dose.C], ["EUCAST_only", "CLSI_only", "both"], "neither")
    print("Dose written in the header vs disk content defined by the guidelines (column x file units):")
    print(cols_dose.groupby("verdict").agg(columns=("n", "size"), results=("n", "sum")).to_string())
    print("\nTOP 'neither' (dose in header matches NO EUCAST/CLSI 2019-2024 disk content -> check header, typo, or non-standard disk):")
    print(cols_dose[cols_dose.verdict == "neither"].groupby(["antibiotic_amr_code", "dose_ug"]).n.sum().sort_values(ascending=False).head(12).to_string())
    exp = DOSES.copy()

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    # ---- file-level guideline inference with an exact binomial test (H0: EUCAST and CLSI equally likely) ----
    GLF = []
    for fid, g in cols_dose[cols_dose.verdict.isin(["EUCAST_only", "CLSI_only"])].groupby("file_id"):
        e, c = int((g.verdict == "EUCAST_only").sum()), int((g.verdict == "CLSI_only").sum()); p = stats.binomtest(max(e, c), e + c, 0.5).pvalue
        GLF.append(dict(file_id=fid, n_decisive=e + c, n_EUCAST=e, n_CLSI=c, p_value=p, family="EUCAST" if e >= c else "CLSI"))
    GLF = pd.DataFrame(GLF)
    INVg = INV[["file_id", "file", "hospital_code", "year"]].merge(GLF, on="file_id", how="left")
    INVg["guideline_family"] = np.where(INVg.n_decisive.fillna(0) >= 3, INVg.family, "EUCAST"); INVg["guideline_basis"] = np.where(INVg.n_decisive.fillna(0) >= 3, "dose evidence", "ASSUMED (no dose evidence)")
    INVg.loc[(INVg.n_decisive >= 3) & (INVg.p_value > 0.05), "guideline_basis"] = "dose evidence (not significant: mixed)"
    INVg["year_for_gl"] = INVg.year.fillna(2020).astype(int)
    INVg["guideline"] = [pick_guideline(f, y) for f, y in zip(INVg.guideline_family, INVg.year_for_gl)]
    INVg.to_csv(AUDIT_DIR / "09_guideline_per_file.csv", index=False)
    print(INVg.groupby(["guideline_family", "guideline_basis"]).size().to_string())
    print("\nfiles where CLSI is detected:"); print(INVg[INVg.guideline_family == "CLSI"][["file", "n_EUCAST", "n_CLSI", "p_value"]].to_string(index=False))
    check("every file has a resolvable guideline", INVg.guideline.notna().all(), INVg[INVg.guideline.isna()].file.tolist()[:3])
    FILE_GL = dict(zip(INVg.file_id, INVg.guideline))

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    # ---- run as_sir: ONLY disk/MIC numeric results of identified antibiotics and organisms; lab-reported S/I/R are NOT re-interpreted ----
    DF_LONG["guideline_used"] = DF_LONG.file_id.astype(object).map(FILE_GL)
    DF_LONG["uti"] = DF_LONG.specimen_en.astype(str).eq("Urine")
    elig = (DF_LONG.result_kind == "antibiotic") & DF_LONG.method_final.isin(["disk", "mic"]) & (DF_LONG.value_type != "disk_out_of_range") \
           & DF_LONG.antibiotic_amr_code.notna() & DF_LONG.organism_amr_code.notna() & DF_LONG.guideline_used.notna()
    print("eligible for as_sir:", pct_ci(int(elig.sum()), int((DF_LONG.result_kind == 'antibiotic').sum())), "of antibiotic results")
    print("not eligible, by reason:")
    for nm, m in (("lab-reported S/I/R (kept as reported)", DF_LONG.method_final == "sir_reported"), ("no organism code", DF_LONG.organism_amr_code.isna()),
                  ("no antibiotic code", DF_LONG.antibiotic_amr_code.isna()), ("disk outside 6-50 mm", DF_LONG.value_type == "disk_out_of_range")):
        print(f"   {nm:42s} {int((m & (DF_LONG.result_kind == 'antibiotic')).sum()):>9d}")
    def fmt_x(r_type, num, op):
        s = ("%g" % num); return s if r_type == "disk" else (("" if op in ("=", "") else op) + s)
    sir_amr = pd.Series(pd.NA, index=DF_LONG.index, dtype=object)
    E = DF_LONG[elig]
    keys = ["guideline_used", "antibiotic_amr_code", "method_final"]
    t0 = time.time(); n_calls = 0
    for (gl, abc, meth), g in E.groupby(keys, observed=True):
        x = [fmt_x(meth, n, o) for n, o in zip(g.value_num, g.value_op)]
        try:
            res = list(AMR.as_sir(x, mo=g.organism_amr_code.astype(str).tolist(), ab=str(abc), guideline=str(gl), uti=g.uti.tolist(), enforce_method=meth))
        except Exception as e:
            print("as_sir failed for", gl, abc, meth, repr(e)[:100]); continue
        sir_amr.loc[g.index] = [None if amr_na(v) else str(v) for v in res]; n_calls += 1
    DF_LONG["sir_amr"] = sir_amr
    print(f"as_sir: {n_calls} calls in {time.time()-t0:.0f}s")
    print(DF_LONG.loc[elig, "sir_amr"].value_counts(dropna=False).to_string())
    na_rate = DF_LONG.loc[elig, "sir_amr"].isna().mean(); print("unclassified among eligible:", pct_ci(int(DF_LONG.loc[elig, 'sir_amr'].isna().sum()), int(elig.sum())))
    nc = DF_LONG[elig & DF_LONG.sir_amr.isna()].groupby(["organism_amr_name", "antibiotic_amr_code", "method_final"], observed=True).size().sort_values(ascending=False).head(20)
    nc.to_csv(AUDIT_DIR / "09_unclassified_top.csv"); print("\nTop unclassified (no breakpoint for organism/antibiotic/guideline):"); print(nc.to_string())
    try:
        H = pd.DataFrame(AMR.sir_interpretation_history()); H.to_excel(AUDIT_DIR / "09_sir_interpretation_history.xlsx", index=False)
        print("\nsir_interpretation_history:", H.shape, H["outcome"].value_counts().to_dict() if "outcome" in H else "")
    except Exception as e: print("sir_interpretation_history unavailable:", repr(e)[:80])
    check("as_sir unclassified share among eligible results <= 15%", na_rate <= 0.15, pct_ci(int(DF_LONG.loc[elig, 'sir_amr'].isna().sum()), int(elig.sum())))

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    # ---- VALIDATION 1: monotonicity (an interpretation can never go S -> R -> S as the value increases) ----
    ord_ = {"S": 0, "I": 1, "R": 2}
    V = DF_LONG[elig & DF_LONG.sir_amr.isin(ord_)][["guideline_used", "antibiotic_amr_code", "method_final", "organism_amr_code", "uti", "value_num", "value_op", "sir_amr"]].copy()
    V["o"] = V.sir_amr.map(ord_)
    viol = 0; ngroups = 0
    for _, g in V.groupby(["guideline_used", "antibiotic_amr_code", "method_final", "organism_amr_code", "uti"], observed=True):
        meth = g.method_final.iloc[0]
        s_ = g.groupby("value_num").o.agg(["min", "max"]).sort_index(ascending=(meth == "mic"))   # susceptible end first
        if len(s_) < 2: continue
        ngroups += 1
        if (s_["max"].values[:-1] > s_["min"].values[1:]).any(): viol += 1
    print(f"groups tested: {ngroups} | monotonicity violations: {viol}")
    check("S/I/R calls are monotone in value (0 violations expected)", viol == 0, f"{viol}/{ngroups} groups")

    # ---- VALIDATION 2: against YOUR own breakpoint table (R vs not-R; 16 organisms) ----
    BPo = D["breakpoints_by_organism"].copy()
    for c in ("disk_R", "disk_R_alt", "cmi_R", "cmi_R_alt"): BPo[c] = pd.to_numeric(BPo[c], errors="coerce")
    own = {(o, a): r for o, a, r in zip(BPo.organism, BPo.antibiotic_en, BPo.to_dict("records"))}
    def own_isR(org, ab_en, meth, num, op, spec):
        r = own.get((org, ab_en))
        if r is None: return np.nan
        cond = str(r["disk_R_alt_condition"] if meth == "disk" else r["cmi_R_alt_condition"]).lower()
        use_alt = (cond == "urine" and spec == "Urine") or (cond == "csf" and spec == "Cerebrospinal fluid")
        if meth == "disk":
            t = r["disk_R_alt"] if (use_alt and pd.notna(r["disk_R_alt"])) else r["disk_R"]
            return np.nan if pd.isna(t) else float(num < t)
        t = r["cmi_R_alt"] if (use_alt and pd.notna(r["cmi_R_alt"])) else r["cmi_R"]
        if pd.isna(t): return np.nan
        return float(num >= t) if op == ">" else float(num > t)
    W = DF_LONG[elig & DF_LONG.sir_amr.isin(ord_) & DF_LONG.organism_amr_name.notna()].copy()
    W["org_t"] = W.organism_amr_name.astype(str).map(lambda n: next((o for o in set(BPo.organism) if tx(o) == tx(n) or tx(o).split()[0] == tx(n).split()[0] and o.endswith("spp")), None))
    W = W[W.org_t.notna()]
    W["ab_en"] = W.antibiotic_amr_name.astype(str)
    W["own_R"] = [own_isR(o, a, m, n, op, s) for o, a, m, n, op, s in zip(W.org_t, W.ab_en, W.method_final, W.value_num, W.value_op, W.specimen_en.astype(str))]
    W = W.dropna(subset=["own_R"]); W["amr_R"] = (W.sir_amr == "R").astype(float)
    if len(W):
        agree = int((W.own_R == W.amr_R).sum()); b = int(((W.amr_R == 1) & (W.own_R == 0)).sum()); c = int(((W.amr_R == 0) & (W.own_R == 1)).sum())
        print(f"R-vs-not-R agreement with your own table: {pct_ci(agree, len(W))} | kappa={cohen_kappa(W.amr_R, W.own_R):.3f} | McNemar p={mcnemar_exact(b, c):.3g} (AMR-R/own-S={b}, AMR-S/own-R={c})")
        pa = W.groupby("ab_en").apply(lambda g: pd.Series({"n": len(g), "agree_%": 100 * (g.own_R == g.amr_R).mean()})).sort_values("agree_%")
        pa.to_csv(AUDIT_DIR / "09_agreement_by_antibiotic.csv"); print("\nlowest agreement by antibiotic (check which guideline version your table follows):"); print(pa.head(10).round(1).to_string())
        check("as_sir agrees with the in-house breakpoint table in >= 90% (R vs non-R)", agree / len(W) >= 0.90, pct_ci(agree, len(W)))
    else: print("no comparable results (organism/antibiotic names differ) - inspect W construction")

## 10. Consolidation — which final dataset: `df_long` or a wide `df`?

**Recommendation (and why)**

| | `df_long` (master) | wide (derived) |
|---|---|---|
| Granularity | isolate × antibiotic × method × dose | isolate (one row) |
| Several tests of the same drug (disk **and** MIC, two doses) | all kept, auditable | must be collapsed by a rule (information loss) |
| Provenance (file, hospital, guideline, `as_sir` vs lab-reported) | per cell | lost or duplicated |
| Sparsity | none (no NaN cells) | ~60% NaN with 100+ columns |
| Adding a new drug/year | new rows only | schema change |
| Use for GLASS / dashboards / models | filter + pivot in one line | ready-made |

So: **`df_long` is the single source of truth (Parquet); the wide table is generated from it with an explicit, logged rule** (MIC > disk > lab-reported when the same isolate has several results for a drug).

In [ ]:
if not AMR_READY:
    print("SKIPPED: AMR package not available in this session (install it, then re-run this cell).")
else:
    rank = {"mic": 0, "disk": 1, "sir_reported": 2}
    import gc
    for _n in ("E", "V", "W"): globals().pop(_n, None)
    gc.collect()
    _lab = (DF_LONG.method_final == "sir_reported").values
    _amr = DF_LONG.sir_amr.notna().values
    _sf = DF_LONG.sir_amr.astype(object)
    _sf = _sf.where(_sf.notna(), DF_LONG.sir_prior.astype(object).where(_lab))
    DF_LONG["sir_final"] = _sf.where(_sf.isin(["S", "I", "R", "SDD", "NI"])).astype("category")
    _codes = np.zeros(len(DF_LONG), dtype="int8"); _codes[_lab] = 2; _codes[_amr] = 1
    DF_LONG["sir_source"] = pd.Categorical.from_codes(_codes, categories=["none", "AMR::as_sir", "reported_by_lab"])
    del _sf, _codes; gc.collect()
    Q = DF_LONG.loc[(DF_LONG.result_kind == "antibiotic") & DF_LONG.sir_final.notna() & DF_LONG.antibiotic_amr_code.notna(), ["row_uid", "antibiotic_amr_code", "method_final", "sir_final"]].copy()
    Q["row_uid"] = Q.row_uid.astype(object); Q["antibiotic_amr_code"] = Q.antibiotic_amr_code.astype(object); Q["sir_final"] = Q.sir_final.astype(object)
    Q["rk"] = Q.method_final.map(rank)
    Q = Q.sort_values(["row_uid", "antibiotic_amr_code", "rk"])
    dupe = Q.duplicated(["row_uid", "antibiotic_amr_code"], keep="first"); print(f"isolate-drug pairs with several results: {int(dupe.sum())} ({100*dupe.mean():.1f}% of interpreted results) -> collapsed by MIC > disk > lab rule in the wide view only")
    conflict = Q[Q.duplicated(["row_uid", "antibiotic_amr_code"], keep=False)].groupby(["row_uid", "antibiotic_amr_code"], observed=True).sir_final.nunique()
    print("of those, pairs whose several results DISAGREE on S/I/R:", int((conflict > 1).sum()), "-> pct %.2f%%" % (100 * (conflict > 1).mean() if len(conflict) else 0))
    WIDE = (Q[~dupe].pivot(index="row_uid", columns="antibiotic_amr_code", values="sir_final"))
    iso_meta = ISO.set_index("row_uid")[["file_id", "hospital_code", "file_year", "patient_key", "sex_std", "birth_year", "collection_date", "ward_en", "population", "specimen_en", "organism_raw", "in_scope", "is_exact_duplicate_row"]]
    DF_WIDE = iso_meta.join(WIDE, how="inner").reset_index()
    DF_WIDE["organism_amr_name"] = DF_WIDE.organism_raw.astype(object).map(lambda v: ORG_NAME.get(str(v).strip()) if pd.notna(v) else None)
    print("df_wide:", DF_WIDE.shape)
    check("wide row count == isolates having >=1 interpreted result", len(DF_WIDE) == Q.row_uid.nunique(), f"{len(DF_WIDE)} vs {Q.row_uid.nunique()}")

In [ ]:
# ---- export ----
KEEP = [c for c in ["row_uid", "file_id", "hospital_code", "file_year", "family", "patient_key", "sex", "birth_year", "collection_date", "ward_raw", "ward_en", "population",
        "ward_src", "specimen_raw", "specimen_en", "specimen_src", "in_scope", "organism_raw", "organism_dict", "organism_amr_code", "organism_amr_name", "organism_source",
        "col_raw", "result_kind", "antibiotic_en", "antibiotic_amr_code", "antibiotic_amr_name", "dose_ug", "method_hint", "method_final", "value_raw", "value_type",
        "value_num", "value_op", "sir_prior", "sir_star", "guideline_used", "uti", "sir_amr", "sir_final", "sir_source", "is_exact_duplicate_row"] if c in DF_LONG.columns]
DF_LONG.drop(columns=[c for c in DF_LONG.columns if c not in KEEP], inplace=True); OUT = DF_LONG
def save(df, stem):
    try:
        df.to_parquet(OUT_DIR / f"{stem}.parquet", index=False); return stem + ".parquet"
    except Exception as e:
        df.to_csv(OUT_DIR / f"{stem}.csv.gz", index=False); return stem + ".csv.gz (pyarrow missing)"
print("saved:", save(OUT, "df_long_final"))
if "DF_WIDE" in globals(): print("saved:", save(DF_WIDE, "df_wide_derived"))
print("saved:", save(LIS, "lis_records"), "|", save(ISO, "isolate_records"))
print("df_long_final:", OUT.shape, "| columns:", len(KEEP))

## 11. Test dashboard (everything above, one table)

In [ ]:
TD = pd.DataFrame(TESTS); TD.to_csv(AUDIT_DIR / "99_test_dashboard.csv", index=False)
major = TD[~TD.test.str.match(r"^F\d{3} ")]
print(major.to_string(index=False))
print(f"\nper-file conservation tests: {int(TD.test.str.match(r'^F\d{3} ').sum())} | failed: {int((~TD[TD.test.str.match(r'^F\d{3} ')]['pass']).sum())}")
print(f"TOTAL: {int(TD['pass'].sum())} passed / {int((~TD['pass']).sum())} FAILED")

## 12. What this pipeline does **not** prove (read before publishing numbers)

1. **Dictionary correctness ≠ completeness.** In-sample coverage is optimistic; the true accuracy needs a blind, stratified expert audit of distinct strings (`audit/07_*`, `audit/08_*`) scored with a one-sided exact binomial test against a ≥ 99% target.
2. **AMR-only matches are proposals.** `as_mo` fuzzy matching can be confidently wrong (`M catar` → *M. tuberculosis* in your earlier run). Every row with `organism_source == "amr_unreviewed"` must be reviewed before it enters a surveillance report.
3. **Guideline per file is inferred**, not known. Confirm EUCAST/CLSI and the version with each laboratory (`audit/09_guideline_per_file.csv`).
4. **Lab-reported S/I/R (≈ 20% of results) were produced under unknown breakpoints** and are kept as-is, flagged `reported_by_lab`.
5. **No first-isolate de-duplication yet.** GLASS requires one isolate per patient / organism / specimen / period; `patient_key` and `is_exact_duplicate_row` are provided for that next step. Patient IDs are only unique **within** a hospital.
6. **`custom` additions to AMR are session-scoped** (re-run in every session).